# Qubit Tune-up with SHF Instruments

In this notebook we demonstrate qubit tuneup with the LabOne Q software, implemented as a sequence of experiments. 

Before starting the experiments, we define a set of initial qubit parameters, as might be known from fabrication. 

These parameters can then be used to update the baseline calibration used in the experiments.

## 0. General Imports and Definitions
### 0.1 Python Imports 

In [ ]:
# LabOne Q:
from laboneq.simple import *

# plotting and fitting functionality
from laboneq.analysis.fitting import (
    lorentzian,
    oscillatory,
    oscillatory_decay,
    exponential_decay,
)
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

# descriptor imports
from laboneq.contrib.example_helpers.descriptors.shfsg_shfqa_pqsc import (
    descriptor_shfsg_shfqa_pqsc,
)

# for saving results and pulse sheets
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np

### 0.2 Function Definitions

In [ ]:
# define sweep parameter
def create_readout_freq_sweep(qubit, start_freq, stop_freq, num_points):
    return LinearSweepParameter(
        uid=f"res_freq_{qubit.uid}",
        start=start_freq + qubit.parameters.readout_resonator_frequency - qubit.parameters.readout_lo_frequency,
        stop=stop_freq + qubit.parameters.readout_resonator_frequency - qubit.parameters.readout_lo_frequency,
        count=num_points,
    )

# function that returns the calibration of the readout line oscillator for spectroscopy
def res_spec_calib(frequency_sweep, amplitude_sweep=None):
    exp_calibration = Calibration()
    # sets the oscillator of the experimental measure signal
    exp_calibration["measure"] = SignalCalibration(
        # for spectroscopy, use the hardware oscillator of the QA, and set the sweep parameter as frequency
        oscillator=Oscillator(
            "readout_osc",
            frequency=frequency_sweep,
            modulation_type=ModulationType.HARDWARE,
        ),
        amplitude=amplitude_sweep,
    )
    return exp_calibration


# signal maps for the qubit readout only
def res_spec_map(qubit):
    signal_map = {
        "measure": device_setup.logical_signal_groups[qubit.uid].logical_signals[
            "measure_line"
        ],
        "acquire": device_setup.logical_signal_groups[qubit.uid].logical_signals[
            "acquire_line"
        ],
    }
    return signal_map

# create gaussian square pulse for readout
def create_readout_pulse(
    qubit, sigma=0.2
):
    readout_pulse = pulse_library.gaussian_square(
        uid=f"readout_pulse_{qubit.uid}",
        length=qubit.parameters.user_defined['readout_len'],
        amplitude=qubit.parameters.user_defined['readout_amp'],
        width=qubit.parameters.user_defined['readout_len'] * 0.9,
        sigma=sigma,
    )
    return readout_pulse

# qubit spectroscopy freq sweep
def create_drive_freq_sweep(qubit, start_freq, stop_freq, num_points):
    return LinearSweepParameter(
        uid=f"drive_freq_{qubit.uid}",
        start=start_freq + qubit.parameters.resonance_frequency_ge - qubit.parameters.drive_lo_frequency,
        stop=stop_freq + qubit.parameters.resonance_frequency_ge - qubit.parameters.drive_lo_frequency,
        count=num_points,
    )

# define square pulse for qubit spec
def create_drive_spec_pulse(qubit, amp = 0.95):
    pulse = pulse_library.const(
        uid=f"drive_spec_pulse_{qubit.uid}",
        length=qubit.parameters.user_defined["pulse_length"],
        amplitude=amp, #max power to start
    )
    return pulse

# signal map for qubit drive and readout
def signal_map_default(qubit):
    signal_map = {
        "drive": device_setup.logical_signal_groups[f"{qubit.uid}"].logical_signals[
            "drive_line"
        ],
        "measure": device_setup.logical_signal_groups[f"{qubit.uid}"].logical_signals[
            "measure_line"
        ],
        "acquire": device_setup.logical_signal_groups[f"{qubit.uid}"].logical_signals[
            "acquire_line"
        ],
    }
    return signal_map

#define gaussian pulse for qubit drive
def create_rabi_drive_pulse(qubit):
    return pulse_library.gaussian(
        uid=f"gaussian_drive_{qubit.uid}",
        length=qubit.parameters.user_defined["pulse_length"],
        amplitude=1,
    )

# define qubit drive rabi sweep
def create_rabi_amp_sweep(qubit, amp_num, uid="rabi_amp"):
    amp_min = 0.1
    amp_max = min([qubit.parameters.user_defined['amplitude_pi'] * 2.2, 1.0])
    return LinearSweepParameter(uid=uid, start=amp_min, stop=amp_max, count=amp_num)

# define delay time sweep for Ramsey, T1, echo
def create_delay_sweep(
    start=0, stop=50e-6, count=100, axis_name="Time [s]"
):
    time_sweep = LinearSweepParameter(
        uid="time_sweep_param", start=start, stop=stop, count=count, axis_name=axis_name
    )
    return time_sweep

# define ramsey drive pulse
def create_ramsey_drive_pulse(qubit):
    return pulse_library.gaussian(
        uid=f"gaussian_drive_{qubit.uid}",
        length=qubit.parameters.user_defined['pulse_length'],
        amplitude=qubit.parameters.user_defined['amplitude_pi'] / 2,
    )

# define T1 drive pulse
def create_T1_drive_pulse(qubit):
    return pulse_library.gaussian(
        uid=f"gaussian_drive_{qubit.uid}",
        length=qubit.parameters.user_defined['pulse_length'],
        amplitude=qubit.parameters.user_defined['amplitude_pi'],
    )

def create_pi_2_pulse(qubit):
    return pulse_library.gaussian(
        uid=f"gaussian_pi_2_drive_{qubit.uid}",
        length=qubit.parameters.user_defined['pulse_length'],
        amplitude=qubit.parameters.user_defined['amplitude_pi']/2,
    )

def create_pi_pulse(qubit):
    return pulse_library.gaussian(
        uid=f"gaussian_pi_drive_{qubit.uid}",
        length=qubit.parameters.user_defined['pulse_length'],
        amplitude=qubit.parameters.user_defined['amplitude_pi'],
    )

## 1. Define the Instrument Setup and Required Experimental Parameters
### 1.1 Create device setup

Create the device setup from the descriptor, and apply some convenient mapping to instruments and logical signals.

In [ ]:
descriptor_shfqc = """
instruments:
  SHFQC:
  - address: DEV12237
    uid: device_shfqc
    interface: 1gbe

connections:
  device_shfqc:
    - iq_signal: q0/drive_line
      ports: SGCHANNELS/0/OUTPUT

    - iq_signal: q0/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - acquire_signal: q0/acquire_line
      ports: [QACHANNELS/0/INPUT]

    - iq_signal: q1/drive_line
      ports: SGCHANNELS/2/OUTPUT

    - iq_signal: q1/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - acquire_signal: q1/acquire_line
      ports: [QACHANNELS/0/INPUT]
"""

In [ ]:
# Define and Load our Device Setup
device_setup = DeviceSetup.from_descriptor(
    yaml_text=descriptor_shfqc,
    server_host="localhost",  # ip address of the LabOne dataserver used to communicate with the instruments
    server_port="8004",  # port number of the dataserver - default is 8004
    setup_name="my_setup",  # setup name
)

# define shortcut to logical signals for convenience
lsg = {
    qubit_name: device_setup.logical_signal_groups[qubit_name].logical_signals
    for qubit_name in device_setup.logical_signal_groups.keys()
}

### 1.2 Qubit Parameters

A python dictionary containing all parameters needed to control and readout the qubits - frequencies, pulse lengths, timings

May initially contain only the design parameters and will be updated with measurement results during the tuneup procedure

In [ ]:
q0 = Transmon.from_logical_signal_group(
    "q0",
    lsg=device_setup.logical_signal_groups["q0"],
    parameters=TransmonParameters(
        resonance_frequency_ge=3.15e9,     #Drive frequency for qubit ge transition
        resonance_frequency_ef=2.95e9,     #Drive frequency for qubit ef transtion
        drive_lo_frequency=6.5e9,          #Center frequency for qubit drive, needs to be within 500 MHz of both ge and ef frequencies
        readout_resonator_frequency=6.040e9, #Readout frequency for readout resonator
        readout_lo_frequency=6.5e9,        #Center frequency for readout line, shared node, all qubits need to have the same value
        readout_integration_delay=60e-9,       #Propogation delay for readout aqcuire line, relative to default 212 ns, shared node, all qubits need to have the same value
        drive_range = 10,                  #Output range for qubit drive in dBm
        readout_range_out = -0,             #Output range for readout resonator in dBm, shared node, all qubits need to have the same value
        readout_range_in = -15,              #Input range for readout resonator in dBm, shared node, all qubits need to have the same value
        
        user_defined={                     #Pulse Parameters to be included as user_defined
            "amplitude_pi": 0.99,           #Amplitude of Pi pulse, linear gain from 0 to 1 of drive_range
            "pulse_length": 50e-9,         #Length of drive/Pi pulse
            "readout_len": 2e-6,           #Length of readout pulse
            "readout_amp": 0.98,           #Amplitude of readout pulse, linear gain from 0 to 1 of readout_range_out, starts at 1 for spectroscopy
            "reset_length": 5e-6,          #Relaxation time to return to ground state
        },
    ),
)

q1 = Transmon.from_logical_signal_group(
    "q1",
    lsg=device_setup.logical_signal_groups["q1"],
    parameters=TransmonParameters(
        resonance_frequency_ge=6.25e9,
        resonance_frequency_ef=5.95e9,
        drive_lo_frequency=7.1e9,
        readout_resonator_frequency=7.4e9,
        readout_lo_frequency=q0.parameters.readout_lo_frequency, #Shared node, needs to match q0 center frequency
        readout_integration_delay=q0.parameters.readout_integration_delay, #Shared node, needs to match q0
        drive_range = 10,
        readout_range_out = q0.parameters.readout_range_out, #Shared node, needs to match q0
        readout_range_in = q0.parameters.readout_range_in, #Shared node, needs to match q0
        user_defined={
            "amplitude_pi": 0.6,
            "pulse_length": 50e-9,
            "readout_len": 2e-6,
            "readout_amp": 0.98,
            "reset_length": 5e-6,
        },
    ),
)

In [ ]:
q0.parameters.resonance_frequency_ge=1300000000.0
q0.parameters.resonance_frequency_ef=2950000000.0
q0.parameters.drive_lo_frequency=1000000000.0
q0.parameters.readout_resonator_frequency=6819000000.0
q0.parameters.readout_lo_frequency=6500000000.0
q0.parameters.readout_integration_delay=6e-08
q0.parameters.drive_range=-15
q0.parameters.readout_range_out=-30
q0.parameters.readout_range_in=-15
q0.parameters.flux_offset_voltage=0
q0.parameters.user_defined={
'amplitude_pi': 0.39,
'pulse_length': 2e-06,
'readout_len': 2e-06,
'readout_amp': 0.3,
'reset_length': 5e-07,}

q1.parameters.resonance_frequency_ge=977580000.0
q1.parameters.resonance_frequency_ef=5950000000.0
q1.parameters.drive_lo_frequency=1400000000.0
q1.parameters.readout_resonator_frequency=6883500000.0
q1.parameters.readout_lo_frequency=6500000000.0
q1.parameters.readout_integration_delay=6e-08
q1.parameters.drive_range=-10
q1.parameters.readout_range_out=-15
q1.parameters.readout_range_in=-15
q1.parameters.flux_offset_voltage=0
q1.parameters.user_defined={
'amplitude_pi': 0.99,
'pulse_length': 1e-06,
'readout_len': 2e-06,
'readout_amp': 0.03,
'reset_length': 5e-07,}

In [ ]:
#Finish calibrating SHFQC with qubit parameters
qubits = [q0, q1]
for qubit in qubits:
    device_setup.set_calibration(qubit.calibration())

#Choose which qubit will be measured
measure_q = q0

### 2.2 Create and Connect to a QCCS Session 

Establishes the connection to the instruments and readies them for experiments

In [ ]:
# perform experiments in emulation mode or on hardware - if True, also generate dummy data for fitting
emulate = False

# create and connect to a session
session = Session(device_setup=device_setup)
session.connect(do_emulation=emulate)

## 3. Qubit Tuneup - Experimental Sequence

Sequence of experiments for tuneup from scratch of a superconducting qubit in circuit QED architecture 

### 3.1 Resonator Spectroscopy: CW

Find the resonance frequency of the qubit readout resonator by looking at the transmission or reflection of a probe signal applied through the readout line

#### 3.1.1 Additional Experimental Parameters

Define the frequency scan

#### 3.1.2 Experiment Definition

Define the experimental pulse and readout sequence - here without any explicit qubit reference

Explicit qubit reference is then given through different experimental calibration and signal maps

In [ ]:
# function that defines a resonator spectroscopy experiment, and takes the frequency sweep as a parameter
def res_spectroscopy_CW(freq_sweep, exp_settings):
    # Create resonator spectroscopy experiment - uses only readout drive and signal acquisition
    exp_spec = Experiment(
        uid="Resonator Spectroscopy",
        signals=[
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define experimental sequence
    # loop - average multiple measurements for each frequency - measurement in spectroscopy mode
    with exp_spec.acquire_loop_rt(
        uid="shots",
        count=exp_settings["num_averages"],
        acquisition_type=AcquisitionType.SPECTROSCOPY,
    ):
        with exp_spec.sweep(uid="res_freq", parameter=freq_sweep):
            # readout pulse and data acquisition
            with exp_spec.section(uid="spectroscopy"):
                # resonator signal readout
                exp_spec.acquire(
                    signal="acquire",
                    handle="res_spec",
                    length=exp_settings["integration_time"],
                )
            with exp_spec.section(uid="delay", length=1e-6):
                # holdoff time after signal acquisition
                exp_spec.reserve(signal="measure")

    return exp_spec

#### 3.1.3 Run and Evaluate Experiment
Runs the experiment and evaluates the data returned by the measurement

# R6, Q0 (F-C0)

In [ ]:
print(measure_q)

In [ ]:
measure_q.parameters.user_defined['readout_amp'] = 0.03

In [ ]:
measure_q.parameters.readout_resonator_frequency = 6.881e9
measure_q.parameters.resonance_frequency_ge = 0.97758e9

In [ ]:
measure_q.parameters.readout_range_out = -15
measure_q.parameters.readout_range_in = -15

In [ ]:
measure_q.parameters.user_defined['pulse_length'] = 2e-06
measure_q.parameters.user_defined['readout_len'] = 2e-06

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

In [ ]:
# frequency range of spectroscopy scan - around expected resonator frequency as defined in qubit parameters
start_freq = -15e6
stop_freq = 15e6
num_points = 201
integration_time = 5e-2
num_averages = 1

In [ ]:
res = (stop_freq + (-start_freq))/(num_points*1e6)
print(f'resolution = {res} MHz')

In [ ]:
# define the experiment with the frequency sweep relevant for qubit 0
freq_sweep = create_readout_freq_sweep(measure_q, start_freq, stop_freq, num_points)
exp_settings = {"integration_time": integration_time, "num_averages": num_averages}
exp_spec = res_spectroscopy_CW(freq_sweep, exp_settings)

# set signal calibration and signal map for experiment
exp_spec.set_calibration(res_spec_calib(freq_sweep))

In [ ]:
exp_spec.set_signal_map(res_spec_map(measure_q))

# run the experiment on the open instrument session
compiled_res_spec = session.compile(exp_spec)
res_spec_results = session.run()

# save the data
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/res_spec_results_{timestamp}.json")
print(f"File saved as Results_Needed/res_spec_results_{timestamp}.json")

In [ ]:
electrical_delay = 60.68e-9
# electrical_delay = 1
# _delay = 10e9

# get the measurement data returned by the instruments from the QCCS session
spec_res = res_spec_results.get_data("res_spec")
# define the frequency axis from the qubit parameters
spec_freq = measure_q.parameters.readout_lo_frequency + res_spec_results.get_axis("res_spec")[0]
# %matplotlib qt
if emulate:
    # create some dummy data if running in emulation mode
    spec_res = lorentzian(
        spec_freq,
        10e6,
        measure_q.parameters.readout_frequency * (0.995 + 0.01 * np.random.rand(1)[0])
        + measure_q.parameters.readout_lo_frequency,
        -1e7,
        10,
    ) + 0.2 * np.random.rand(len(spec_freq))

# plot the measurement data
fig, [ax1, ax2, ax3] = plt.subplots(3, 1, figsize=(6, 7))
ax1.plot(spec_freq / 1e9, abs(spec_res), ".k")

ax2.plot(spec_freq / 1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res), "orange")
ax3.plot(spec_freq / 1e9, np.unwrap(np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res)), "orange")

ax1.set_ylabel("A (a.u.)")
# ax1.set_title('Mag')

ax2.set_ylabel("$\\phi$ (rad)")
# ax2.set_title('Wrapped phase')

ax3.set_ylabel("$\\phi$ (rad)")
# ax3.set_title('Unwrapped phase')
ax3.set_xlabel("Frequency (GHz)")

vl = []
for _vl in vl:
    ax1.axvline(vl, ls='-', color='tab:blue')
    ax2.axvline(vl, ls='-', color='tab:blue')
    ax3.axvline(vl, ls='-', color='tab:blue')

plt.tight_layout()
plt.show()

## Resonator vs flux 

### resonator spectroscopy setup

In [ ]:
# frequency range of spectroscopy scan -
# around expected resonator frequency as defined in qubit parameters
start_freq = -20e6
stop_freq = 15e6
num_points = 71

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 10

readout_pulse = create_readout_pulse(measure_q)

In [ ]:
2**10

In [ ]:
print(measure_q)

#### 3.2.2 Experiment Definition

In [ ]:
# function that defines a resonator spectroscopy experiment, and takes the frequency sweep as a parameter

def res_spectroscopy_pulsed(freq_sweep, num_averages, readout_pulse):
    # Create resonator spectroscopy experiment - uses only readout drive and signal acquisition
    exp_spec_pulsed = Experiment(
        uid="Resonator Spectroscopy",
        signals=[
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define experimental sequence
    # outer loop - vary drive frequency

    # inner loop - average multiple measurements for each frequency - measurement in spectroscopy mode
    with exp_spec_pulsed.acquire_loop_rt(
        uid="shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.SPECTROSCOPY,
        averaging_mode=AveragingMode.SEQUENTIAL,
    ):
        with exp_spec_pulsed.sweep(
            uid="res_freq",
            parameter=freq_sweep,
            chunk_count=1,
        ):
            # readout pulse and data acquisition
            with exp_spec_pulsed.section(uid="spectroscopy"):
                # play resonator excitation pulse
                exp_spec_pulsed.play(signal="measure", pulse=readout_pulse)
                # resonator signal readout
                exp_spec_pulsed.acquire(
                    signal="acquire", handle="res_spec_pulsed", length=readout_pulse.length
                )
            with exp_spec_pulsed.section(uid="delay", length=10e-6):
                # holdoff time after signal acquisition - minimum 1us required for data processing on UHFQA
                exp_spec_pulsed.reserve(signal="measure")

    return exp_spec_pulsed

#### 3.2.3 Apply Experiment Parameters and Compile

In [ ]:
# measure_q.parameters.readout_range_out = 5

# # apply calibration to device setup
device_setup.set_calibration(
    measure_q.calibration()
)

In [ ]:
# create freq sweep
freq_sweep = create_readout_freq_sweep(measure_q, start_freq, stop_freq, num_points)

# define the experiment with the frequency sweep relevant for qubit
exp_spec_pulsed = res_spectroscopy_pulsed(freq_sweep, num_averages, readout_pulse)

# set signal calibration and signal map for experiment to qubit
exp_spec_pulsed.set_calibration(res_spec_calib(freq_sweep))
exp_spec_pulsed.set_signal_map(res_spec_map(measure_q))

In [ ]:
# compile the experiment on the open instrument session
compiled_spec_pulsed = session.compile(exp_spec_pulsed)

Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)

# generate a pulse sheet to inspect experiment before runtime
show_pulse_sheet("Pulse_Sheets/Pulsed_Spectroscopy", compiled_spec_pulsed)

#### 3.2.4 Run and Evaluate Experiment

In [ ]:
# run the experiment on the open instrument session
spec_pulsed_results = session.run()

# save the data
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/spec_pulsed_results_{timestamp}.json")
print(f"File saved as Results_Needed/spec_pulsed_results_{timestamp}.json")

In [ ]:
electrical_delay = 60.64e-9
# electrical_delay = 1
# _delay = 10e9

# get the measurement data returned by the instruments from the QCCS session
spec_res = spec_pulsed_results.get_data("res_spec_pulsed")
# define the frequency axis from the qubit parameters
spec_freq = measure_q.parameters.readout_lo_frequency + spec_pulsed_results.get_axis("res_spec_pulsed")[0]
# %matplotlib qt
if emulate:
    # create some dummy data if running in emulation mode
    spec_res = lorentzian(
        spec_freq,
        10e6,
        measure_q.parameters.readout_frequency * (0.995 + 0.01 * np.random.rand(1)[0])
        + measure_q.parameters.readout_lo_frequency,
        -1e7,
        10,
    ) + 0.2 * np.random.rand(len(spec_freq))

# plot the measurement data
fig, [ax1, ax2, ax3] = plt.subplots(3, 1, figsize=(6, 7))
ax1.plot(spec_freq / 1e9, abs(spec_res), ".k")

ax2.plot(spec_freq / 1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res), "orange")
ax3.plot(spec_freq / 1e9, np.unwrap(np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res)), "orange")

ax1.set_ylabel("A (a.u.)")
# ax1.set_title('Mag')

ax2.set_ylabel("$\\phi$ (rad)")
# ax2.set_title('Wrapped phase')

ax3.set_ylabel("$\\phi$ (rad)")
# ax3.set_title('Unwrapped phase')
ax3.set_xlabel("Frequency (GHz)")

vl = [6.8835]
for _vl in vl:
    ax1.axvline(_vl, ls='-', 
                # color='tab:blue',
               )
    ax2.axvline(_vl, ls='-', 
                # color='tab:blue',
               )
    ax3.axvline(_vl, ls='-', 
                # color='tab:blue',
               )

plt.tight_layout()

plt.axvline(measure_q.parameters.readout_resonator_frequency*1e-9, alpha=0.4, ls='--', color='tab:purple')

In [ ]:
measure_q.parameters.readout_resonator_frequency = 6.8835e9
device_setup.set_calibration(
    measure_q.calibration()
)

In [ ]:
print(measure_q)

### f_r vs phi ext

In [ ]:
del dc

In [ ]:
from qcodes.instrument_drivers.yokogawa.GS200 import GS200
dc_q1 = GS200('yoko_q1', address = 'GPIB00::15::INSTR')

In [ ]:
dc_q1.off()
dc_q1.output('off')
dc_q1.source_mode('CURR')
dc_q1.ramp_current(0e-3, 1e-6,0)

In [ ]:
dc_q1.output('off')
dc_q1.source_mode('CURR')

In [ ]:
import sys

In [ ]:
dc_q1.ramp_current(0, 1e-6, 0)

In [ ]:
dc_q1.output('on')

In [ ]:
dc_q1.ramp_current(0e-3, 1e-6, 0)

In [ ]:
plot = False  # plot individual resonator spectroscopy
dc_q1.current_range(0.2)

start_curr = 3.0e-3
stop_curr = 15e-3
current_sweep = np.linspace(start_curr,stop_curr, 151)

dc_q1.ramp_current(0e-3, 1e-6, 0)

dc_q1.output('off')
dc_q1.source_mode('CURR')

dc_q1.output('on')
sweep_rspec_results = []
for i, current in enumerate(current_sweep):
    print('step ' + str(i))
    print(f'current: {current}')
    
    dc_q1.ramp_current(current, 0.5e-6, 0)

    # debugging
    # dc.ramp_current(0e-3, 1e-6,0)
    # dc.output('off')
    # sys.exit()
    # time.sleep(0.5)
    
    # run the experiment on qubit 0
    rspec_results = session.run()

    # save the data
    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    session.save_results(f"Results_Needed/spec_pulsed_results_{timestamp}.json")
    print(f"File saved as Results_Needed/spec_pulsed_results_{timestamp}.json")
    
    # get the measurement data returned by the instruments from the QCCS session
    spec_res = rspec_results.get_data("res_spec_pulsed")
    # define the frequency axis from the qubit parameters
    spec_freq = (
        measure_q.parameters.readout_lo_frequency + spec_pulsed_results.get_axis("res_spec_pulsed")[0]
    )
    sweep_rspec_results.append(spec_res)

    # plot the measurement data
    if plot:
        plt.figure()
        fig, [ax1, ax2] = plt.subplots(2, 1)
        ax1.plot(spec_freq / 1e9, abs(spec_res), ".k")
        ax2.plot(spec_freq / 1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res), "orange")
        ax1.set_ylabel("A (a.u.)")
        ax2.set_ylabel("$\\phi$ (rad)")
        ax2.set_xlabel("Frequency (GHz)")
        ax2.axvline(spec_freq[np.argmin(abs(spec_res))]/1e9)

In [ ]:
plt.pcolor( current_sweep*1e3,spec_freq/1e9, np.abs(sweep_rspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title(r'$f_r$ vs $\Phi_{\text{ext}}$, Magnitude', size=10)
plt.colorbar().ax.set_title('LogMag [arb.]', size=10)

In [ ]:
plt.pcolor( current_sweep*1e3,spec_freq/1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*sweep_rspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title('phase')
plt.title(r'$f_r$ vs $\Phi_{\text{ext}}$, Phase', size=10)
plt.colorbar().ax.set_title('Phase [rad]', size=10)
# plt.axvline(8.8, c='tab:blue', alpha=0.4)
# plt.axhline(6.818, c='tab:purple', alpha=0.4)

# fine resonator spectroscopy around half flux

In [ ]:
plot = False  # plot individual resonator spectroscopy
dc_q1.current_range(0.2)

start_curr = 5.0e-3
stop_curr = 9e-3
current_sweep = np.linspace(start_curr,stop_curr, 351)

dc_q1.ramp_current(0e-3, 1e-6, 0)

dc_q1.output('off')
dc_q1.source_mode('CURR')

dc_q1.output('on')
sweep_rspec_results = []
for i, current in enumerate(current_sweep):
    print('step ' + str(i))
    print(f'current: {current}')
    
    dc_q1.ramp_current(current, 0.5e-6, 0)

    # debugging
    # dc.ramp_current(0e-3, 1e-6,0)
    # dc.output('off')
    # sys.exit()
    # time.sleep(0.5)
    
    # run the experiment on qubit 0
    rspec_results = session.run()

    # save the data
    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    session.save_results(f"Results_Needed/spec_pulsed_results_{timestamp}.json")
    print(f"File saved as Results_Needed/spec_pulsed_results_{timestamp}.json")
    
    # get the measurement data returned by the instruments from the QCCS session
    spec_res = rspec_results.get_data("res_spec_pulsed")
    # define the frequency axis from the qubit parameters
    spec_freq = (
        measure_q.parameters.readout_lo_frequency + spec_pulsed_results.get_axis("res_spec_pulsed")[0]
    )
    sweep_rspec_results.append(spec_res)

    # plot the measurement data
    if plot:
        plt.figure()
        fig, [ax1, ax2] = plt.subplots(2, 1)
        ax1.plot(spec_freq / 1e9, abs(spec_res), ".k")
        ax2.plot(spec_freq / 1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res), "orange")
        ax1.set_ylabel("A (a.u.)")
        ax2.set_ylabel("$\\phi$ (rad)")
        ax2.set_xlabel("Frequency (GHz)")
        ax2.axvline(spec_freq[np.argmin(abs(spec_res))]/1e9)

dc_q1.ramp_current(0e-3, 0.2e-6, 0)

In [ ]:
plt.pcolor( current_sweep*1e3,spec_freq/1e9, np.abs(sweep_rspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title(r'$f_r$ vs $\Phi_{\text{ext}}$, Magnitude', size=10)
plt.colorbar().ax.set_title('LogMag [arb.]', size=10)

In [ ]:
plt.pcolor( current_sweep*1e3,spec_freq/1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*sweep_rspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title('phase')
plt.title(r'$f_r$ vs $\Phi_{\text{ext}}$, Phase', size=10)
plt.colorbar().ax.set_title('Phase [rad]', size=10)
plt.axvline(7.0, c='tab:blue', alpha=0.4)
plt.axhline(6.883, c='tab:purple', alpha=0.4)

# Two-tone

In [ ]:
print(q0)

In [ ]:
from copy import deepcopy

In [ ]:
# print(q1)
q1.parameters.resonance_frequency_ge = deepcopy(q0.parameters.resonance_frequency_ge)
q1.parameters.drive_lo_frequency = deepcopy(q0.parameters.drive_lo_frequency)
q1.parameters.readout_resonator_frequency = deepcopy(q0.parameters.readout_resonator_frequency)
q1.parameters.readout_lo_frequency = deepcopy(q0.parameters.readout_lo_frequency)
q1.parameters.readout_integration_delay = deepcopy(q0.parameters.readout_integration_delay)
q1.parameters.drive_range = deepcopy(q0.parameters.drive_range)
q1.parameters.readout_range_out = deepcopy(q0.parameters.readout_range_out)
q1.parameters.readout_range_in = deepcopy(q0.parameters.readout_range_in)
q1.parameters.user_defined = deepcopy(q0.parameters.user_defined)
print(q1)

In [ ]:
measure_q = q1

In [ ]:
dc_q1.ramp_current(7.0e-3, 1e-6, 0)

In [ ]:
dc.output('on')

In [ ]:
# frequency range of spectroscopy scan - defined +/- expected qubit frequency as defined in qubit parameters
# qspec_range = 500e6
# how many frequency points to measure
qspec_num = 501
# qspec_num = 201

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 11

In [ ]:
### QB spec
measure_q.parameters.drive_lo_frequency = 1.4e9

LF_path = False
start_freq = -50e6
stop_freq = 950e6
# measure_q.parameters.resonance_frequency_ge = 1.161e9
# measure_q.parameters.user_defined['amplitude_pi'] = 0.99
# measure_q.parameters.user_defined['amplitude_pi'] = 0.01

### RO spec
# measure_q.parameters.readout_resonator_frequency = 6.816e9
# measure_q.parameters.readout_range_out = -10
# measure_q.parameters.readout_range_in = -15
# measure_q.parameters.user_defined['readout_amp'] = 0.3

# measure_q.parameters.drive_lo_frequency = 7.5e9
# measure_q.parameters.resonance_frequency_ge = 7.5e9

measure_q.parameters.drive_range = -10
# measure_q.parameters.drive_range = 0
# measure_q.parameters.user_defined['reset_length'] = 100e-6
measure_q.parameters.user_defined['reset_length'] = 0.5e-6
# measure_q.parameters.user_defined['pulse_length'] = 1000e-09
measure_q.parameters.user_defined['pulse_length'] = 1000e-09

# measure_q.parameters.user_defined['readout_range_out'] = -10
# measure_q.parameters.user_defined['readout_range_in'] = -10

print(measure_q)

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

#### 3.5.2 Experiment Definition

The frequency sweep of the drive line can now be done in real time (was: near time in older software releases)

In [ ]:
# function that returns a qubit spectroscopy experiment- accepts frequency sweep range as parameter


def qubit_spectroscopy(freq_sweep, drive_pulse, readout_pulse, reset_delay):
    # Create qubit spectroscopy Experiment - uses qubit drive, readout drive and data acquisition lines
    exp_qspec = Experiment(
        uid="Qubit Spectroscopy",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    
    
    # inner loop - real-time averaging - QA in integration mode
    with exp_qspec.acquire_loop_rt(
        uid="freq_shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        
        with exp_qspec.sweep(uid="qfreq_sweep", parameter=freq_sweep):
            # qubit drive
            with exp_qspec.section(uid="qubit_excitation"):
                # exp_qspec.play(signal="drive", pulse=drive_pulse)
                exp_qspec.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
            with exp_qspec.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_qspec.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_qspec.acquire(
                    signal="acquire",
                    handle="qb_spec",
                    kernel=readout_pulse,
                )
            with exp_qspec.section(uid="delay"):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_qspec.delay(signal="measure", time=reset_delay)

    return exp_qspec

In [ ]:
freq_sweep_q = create_drive_freq_sweep(measure_q, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q = Calibration()

if LF_path == True:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
# define experiment with frequency sweep for qubit 0
drive_pulse = create_drive_spec_pulse(measure_q, amp=0.4)
# drive_pulse = create_rabi_drive_pulse(measure_q)

readout_pulse = create_readout_pulse(measure_q)

#update default calibration to qubit settings
device_setup.set_calibration(
    measure_q.calibration()
)

exp_qspec = qubit_spectroscopy(freq_sweep_q, drive_pulse, readout_pulse, reset_delay = measure_q.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec.set_calibration(exp_calibration_q)
exp_qspec.set_signal_map(signal_map_default(measure_q))

In [ ]:
measure_q.parameters.user_defined['reset_length'], freq_sweep_q, drive_pulse, readout_pulse

In [ ]:
# compile the experiment on the open instrument session
compiled_qspec = session.compile(exp_qspec)

#Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)
# generate a pulse sheet to inspect experiment before runtime
#show_pulse_sheet("Pulse_Sheets/Qubit_Spectroscopy", compiled_qspec)

# plot_simulation(compiled_qspec, 0, 10e-6)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
emulate=False

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency

if emulate:
    # create some dummy data if running in emulation mode
    qspec_res = lorentzian(
        qspec_freq,
        5e6,
        measure_q.parameters.resonance_frequency_ge * (0.995 + 0.01 * np.random.rand(1)[0]),
        -2e6,
        1,
    ) + 0.1 * np.random.rand(len(qspec_freq))


# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")

# plt.axvline(1.162, linestyle = '--', alpha=0.4)
# plt.axvline(1.14, linestyle = '--', alpha=0.4)
# plt.axvline(1.191, linestyle = '--', alpha=0.4)
# plt.axvline(1.160, linestyle = '--', alpha=0.4, color='tab:purple')
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

In [ ]:
print(measure_q)

In [ ]:
measure_q.parameters.resonance_frequency_ge = 1.1607e9
device_setup.set_calibration(
    measure_q.calibration()
)

In [ ]:
print(measure_q)

## Two-tone vs flux
Fine two tone sweep to find half-flux

In [ ]:
# dc.current_range(1e-3)
start = 5.8e-3
stop = 6.0e-3

current_sweep = np.linspace(start, stop, 21)
dc_q1.output('on')
sweep_qspec_results = []
for current in current_sweep:
    dc_q1.ramp_current(current, 1e-6, 0)
    time.sleep(0.5)
    # run the experiment on qubit 0
    qspec_results = session.run()

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    qspec_res = qspec_results.get_data("qb_spec")
    qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency
    sweep_qspec_results.append(qspec_res)

    # plot measurement data
    fig = plt.figure()
    plt.plot(qspec_freq / 1e9, abs(qspec_res), ".k")
    plt.ylabel("A (a.u.)")
    plt.xlabel("Frequency (GHz)")

# plt.show()
# dc.ramp_current(5.5e-3,1e-5,0)

In [ ]:
plt.pcolor( current_sweep*1e3,qspec_freq/1e9, np.abs(sweep_qspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')
# plt.ylim(7.19, 7.215)
# plt.clim(-0.0014, 0.0016)
# plt.axvline(6.2)
# current_sweep[8]*1e3

In [ ]:
angle_data = np.unwrap(np.angle(sweep_qspec_results).T)
angle_data[(angle_data > 1)] = np.nan
norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
norm = np.abs(norm)

In [ ]:
# for col in norm.T:
#     plt.figure()
#     plt.plot(col)

In [ ]:
# plt.pcolor( current_sweep*1e3,qspec_freq/1e9, angle_data)
plt.pcolor( current_sweep*1e3,qspec_freq/1e9, norm)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('phase')
# plt.ylim(7.19, 7.215)
# plt.clim(-0.0014, 0.0016)
# plt.axvline(08.84)
# plt.axhline(1.1629, color='tab:orange')
# current_sweep[8]*1e3

In [ ]:
ams = []
ms = []
for col in norm.T:
    ams.append(np.nanargmax(col))
    ms.append(np.nanmax(col))

In [ ]:
peaks_loc = np.nanargmax(norm, axis=0)

In [ ]:
peak_freq = (qspec_freq/1e9)[peaks_loc]

In [ ]:
plt.plot(current_sweep*1e3, peak_freq)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title('01 transition vs flux')
plt.margins(x=0, y=0)

In [ ]:
from scipy import stats
slope, intercept, r_value, p_value, std_err = stats.linregress(current_sweep*1e3,peak_freq)

In [ ]:
slope

## Move Q0 to linear fluxon regime

In [ ]:
q0.parameters.resonance_frequency_ge=1162600000.0
q0.parameters.resonance_frequency_ef=2950000000.0
q0.parameters.drive_lo_frequency=1300000000.0
q0.parameters.readout_resonator_frequency=6816000000.0
q0.parameters.readout_lo_frequency=6500000000.0
q0.parameters.readout_integration_delay=6e-08
q0.parameters.drive_range=5
q0.parameters.readout_range_out=-30
q0.parameters.readout_range_in=-15
q0.parameters.flux_offset_voltage=0
q0.parameters.user_defined={
'amplitude_pi': 0.6,
'pulse_length': 4e-06,
'readout_len': 2e-06,
'readout_amp': 0.3,
'reset_length': 0.0001,
}

In [ ]:
print(q0)

In [ ]:
measure_q = q0

In [ ]:
from qcodes.instrument_drivers.yokogawa.GS200 import GS200
dc_q0 = GS200('yoko_q0', address = 'TCPIP0::192.168.3.193::inst0::INSTR')

### resonator spectroscopy

In [ ]:
# frequency range of spectroscopy scan -
# around expected resonator frequency as defined in qubit parameters
start_freq = -20e6
stop_freq = 15e6
num_points = 71

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 10

readout_pulse = create_readout_pulse(measure_q)

In [ ]:
2**10

In [ ]:
print(measure_q)

#### 3.2.2 Experiment Definition

In [ ]:
# function that defines a resonator spectroscopy experiment, and takes the frequency sweep as a parameter

def res_spectroscopy_pulsed(freq_sweep, num_averages, readout_pulse):
    # Create resonator spectroscopy experiment - uses only readout drive and signal acquisition
    exp_spec_pulsed = Experiment(
        uid="Resonator Spectroscopy",
        signals=[
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define experimental sequence
    # outer loop - vary drive frequency

    # inner loop - average multiple measurements for each frequency - measurement in spectroscopy mode
    with exp_spec_pulsed.acquire_loop_rt(
        uid="shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.SPECTROSCOPY,
        averaging_mode=AveragingMode.SEQUENTIAL,
    ):
        with exp_spec_pulsed.sweep(
            uid="res_freq",
            parameter=freq_sweep,
            chunk_count=1,
        ):
            # readout pulse and data acquisition
            with exp_spec_pulsed.section(uid="spectroscopy"):
                # play resonator excitation pulse
                exp_spec_pulsed.play(signal="measure", pulse=readout_pulse)
                # resonator signal readout
                exp_spec_pulsed.acquire(
                    signal="acquire", handle="res_spec_pulsed", length=readout_pulse.length
                )
            with exp_spec_pulsed.section(uid="delay", length=10e-6):
                # holdoff time after signal acquisition - minimum 1us required for data processing on UHFQA
                exp_spec_pulsed.reserve(signal="measure")

    return exp_spec_pulsed

#### 3.2.3 Apply Experiment Parameters and Compile

In [ ]:
# measure_q.parameters.readout_range_out = 5

# # apply calibration to device setup
device_setup.set_calibration(
    measure_q.calibration()
)

In [ ]:
# create freq sweep
freq_sweep = create_readout_freq_sweep(measure_q, start_freq, stop_freq, num_points)

# define the experiment with the frequency sweep relevant for qubit
exp_spec_pulsed = res_spectroscopy_pulsed(freq_sweep, num_averages, readout_pulse)

# set signal calibration and signal map for experiment to qubit
exp_spec_pulsed.set_calibration(res_spec_calib(freq_sweep))
exp_spec_pulsed.set_signal_map(res_spec_map(measure_q))

In [ ]:
# compile the experiment on the open instrument session
compiled_spec_pulsed = session.compile(exp_spec_pulsed)

Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)

# generate a pulse sheet to inspect experiment before runtime
show_pulse_sheet("Pulse_Sheets/Pulsed_Spectroscopy", compiled_spec_pulsed)

#### 3.2.4 Run and Evaluate Experiment

In [ ]:
# run the experiment on the open instrument session
spec_pulsed_results = session.run()

# save the data
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/spec_pulsed_results_{timestamp}.json")
print(f"File saved as Results_Needed/spec_pulsed_results_{timestamp}.json")

In [ ]:
electrical_delay = 60.68e-9
# electrical_delay = 1
# _delay = 10e9

# get the measurement data returned by the instruments from the QCCS session
spec_res = spec_pulsed_results.get_data("res_spec_pulsed")
# define the frequency axis from the qubit parameters
spec_freq = measure_q.parameters.readout_lo_frequency + spec_pulsed_results.get_axis("res_spec_pulsed")[0]
# %matplotlib qt
if emulate:
    # create some dummy data if running in emulation mode
    spec_res = lorentzian(
        spec_freq,
        10e6,
        measure_q.parameters.readout_frequency * (0.995 + 0.01 * np.random.rand(1)[0])
        + measure_q.parameters.readout_lo_frequency,
        -1e7,
        10,
    ) + 0.2 * np.random.rand(len(spec_freq))

# plot the measurement data
fig, [ax1, ax2, ax3] = plt.subplots(3, 1, figsize=(6, 7))
ax1.plot(spec_freq / 1e9, abs(spec_res), ".k")

ax2.plot(spec_freq / 1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res), "orange")
ax3.plot(spec_freq / 1e9, np.unwrap(np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res)), "orange")

ax1.set_ylabel("A (a.u.)")
# ax1.set_title('Mag')

ax2.set_ylabel("$\\phi$ (rad)")
# ax2.set_title('Wrapped phase')

ax3.set_ylabel("$\\phi$ (rad)")
# ax3.set_title('Unwrapped phase')
ax3.set_xlabel("Frequency (GHz)")

vl = [6.819]
for _vl in vl:
    ax1.axvline(_vl, ls='--', 
                color='tab:purple',
                alpha=0.4,
               )
    ax2.axvline(_vl, ls='--', 
                color='tab:purple',
                alpha=0.4,
               )
    ax3.axvline(_vl, ls='--', 
                color='tab:purple',
                alpha=0.4,
               )

plt.tight_layout()

plt.axvline(measure_q.parameters.readout_resonator_frequency*1e-9)

In [ ]:
measure_q.parameters.readout_resonator_frequency = 6.819e9

### two tone

In [ ]:
dc_q0.ramp_current(7.95e-3, 1e-6, 0)

In [ ]:
# frequency range of spectroscopy scan - defined +/- expected qubit frequency as defined in qubit parameters
# qspec_range = 500e6
# how many frequency points to measure
qspec_num = 401

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 13

In [ ]:
### QB spec
measure_q.parameters.drive_lo_frequency = 1.0e9

LF_path = False
start_freq = -50e6
stop_freq = 50e6
measure_q.parameters.resonance_frequency_ge = 0.71e9
# measure_q.parameters.user_defined['amplitude_pi'] = 0.99
# measure_q.parameters.user_defined['amplitude_pi'] = 0.01

### RO spec
# measure_q.parameters.readout_resonator_frequency = 6.816e9
# measure_q.parameters.readout_range_out = -10
# measure_q.parameters.readout_range_in = -15
# measure_q.parameters.user_defined['readout_amp'] = 0.3

# measure_q.parameters.drive_lo_frequency = 7.5e9
# measure_q.parameters.resonance_frequency_ge = 7.5e9

measure_q.parameters.drive_range = -20
# measure_q.parameters.drive_range = 0
# measure_q.parameters.user_defined['reset_length'] = 100e-6
measure_q.parameters.user_defined['reset_length'] = 0.5e-6
# measure_q.parameters.user_defined['pulse_length'] = 1000e-09
measure_q.parameters.user_defined['pulse_length'] = 4000e-09

# measure_q.parameters.user_defined['readout_range_out'] = -10
# measure_q.parameters.user_defined['readout_range_in'] = -10

print(measure_q)

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

#### 3.5.2 Experiment Definition

The frequency sweep of the drive line can now be done in real time (was: near time in older software releases)

In [ ]:
# function that returns a qubit spectroscopy experiment- accepts frequency sweep range as parameter


def qubit_spectroscopy(freq_sweep, drive_pulse, readout_pulse, reset_delay):
    # Create qubit spectroscopy Experiment - uses qubit drive, readout drive and data acquisition lines
    exp_qspec = Experiment(
        uid="Qubit Spectroscopy",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    
    
    # inner loop - real-time averaging - QA in integration mode
    with exp_qspec.acquire_loop_rt(
        uid="freq_shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        
        with exp_qspec.sweep(uid="qfreq_sweep", parameter=freq_sweep):
            # qubit drive
            with exp_qspec.section(uid="qubit_excitation"):
                # exp_qspec.play(signal="drive", pulse=drive_pulse)
                exp_qspec.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
            with exp_qspec.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_qspec.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_qspec.acquire(
                    signal="acquire",
                    handle="qb_spec",
                    kernel=readout_pulse,
                )
            with exp_qspec.section(uid="delay"):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_qspec.delay(signal="measure", time=reset_delay)

    return exp_qspec

In [ ]:
freq_sweep_q = create_drive_freq_sweep(measure_q, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q = Calibration()

if LF_path == True:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
# define experiment with frequency sweep for qubit 0
drive_pulse = create_drive_spec_pulse(measure_q, amp=0.4)
# drive_pulse = create_rabi_drive_pulse(measure_q)

readout_pulse = create_readout_pulse(measure_q)

#update default calibration to qubit settings
device_setup.set_calibration(
    measure_q.calibration()
)

exp_qspec = qubit_spectroscopy(freq_sweep_q, drive_pulse, readout_pulse, reset_delay = measure_q.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec.set_calibration(exp_calibration_q)
exp_qspec.set_signal_map(signal_map_default(measure_q))

In [ ]:
measure_q.parameters.user_defined['reset_length'], freq_sweep_q, drive_pulse, readout_pulse

In [ ]:
# compile the experiment on the open instrument session
compiled_qspec = session.compile(exp_qspec)

#Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)
# generate a pulse sheet to inspect experiment before runtime
#show_pulse_sheet("Pulse_Sheets/Qubit_Spectroscopy", compiled_qspec)

# plot_simulation(compiled_qspec, 0, 10e-6)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
emulate=False

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency

if emulate:
    # create some dummy data if running in emulation mode
    qspec_res = lorentzian(
        qspec_freq,
        5e6,
        measure_q.parameters.resonance_frequency_ge * (0.995 + 0.01 * np.random.rand(1)[0]),
        -2e6,
        1,
    ) + 0.1 * np.random.rand(len(qspec_freq))


# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")

plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

In [ ]:
measure_q.parameters.resonance_frequency_ge = 0.711e9

## Two-tone vs flux
Fine two tone sweep to find half-flux

In [ ]:
# dc.current_range(1e-3)
start = 7.85e-3
stop = 8.05e-3

current_sweep = np.linspace(start, stop, 41)
dc_q0.output('on')
sweep_qspec_results = []
for current in current_sweep:
    dc_q0.ramp_current(current, 1e-6, 0)
    time.sleep(0.5)
    # run the experiment on qubit 0
    qspec_results = session.run()

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    qspec_res = qspec_results.get_data("qb_spec")
    qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency
    sweep_qspec_results.append(qspec_res)

    # plot measurement data
    # fig = plt.figure()
    # plt.plot(qspec_freq / 1e9, abs(qspec_res), ".k")
    # plt.ylabel("A (a.u.)")
    # plt.xlabel("Frequency (GHz)")

# plt.show()
# dc.ramp_current(5.5e-3,1e-5,0)

In [ ]:
plt.pcolor( current_sweep*1e3,qspec_freq/1e9, np.abs(sweep_qspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')
# plt.ylim(7.19, 7.215)
# plt.clim(-0.0014, 0.0016)
# plt.axvline(6.2)
# current_sweep[8]*1e3

In [ ]:
angle_data = np.abs(np.unwrap(np.angle(sweep_qspec_results).T))
# for col in angle_data.T:
#     plt.figure()
#     plt.plot(col)

In [ ]:
thresh_low = -2
thresh_high = 2
norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
norm[(norm > thresh_high) | (norm < thresh_low)] = np.nan

In [ ]:
# plt.pcolor( current_sweep*1e3,qspec_freq/1e9, angle_data)
plt.pcolor( current_sweep*1e3,qspec_freq/1e9, norm)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('phase')
# plt.ylim(7.19, 7.215)
# plt.clim(-0.0014, 0.0016)
plt.axvline(7.95)
plt.axhline(0.71, color='tab:orange')
# current_sweep[8]*1e3

In [ ]:
for i, col in enumerate(norm.T):
    plt.figure()
    plt.plot(qspec_freq/1e9, col)
    plt.xlabel('Frequency [GHz]')
    plt.ylabel('Digitizer voltage (arb.)')
    plt.margins(x=0)
    plt.title(f'Current = {np.round(current_sweep[i]*1e3, 4)} mA')

In [ ]:
peaks_loc = np.nanargmin(norm, axis=0)

In [ ]:
peak_freq = (qspec_freq/1e9)[peaks_loc]

In [ ]:
plt.plot(current_sweep*1e3, peak_freq)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title('01 transition vs flux')
plt.margins(x=0, y=0)
plt.axvline(7.95)
plt.axhline(0.71, color='tab:orange')

In [ ]:
from scipy import stats
slope, intercept, r_value, p_value, std_err = stats.linregress(current_sweep*1e3,peak_freq)

In [ ]:
slope

## Linear fluxon

### two tone

In [ ]:
dc_q0.ramp_current(7.95e-3, 1e-6, 0)

In [ ]:
# frequency range of spectroscopy scan - defined +/- expected qubit frequency as defined in qubit parameters
# qspec_range = 500e6
# how many frequency points to measure
qspec_num = 1001

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 13

In [ ]:
### QB spec
measure_q.parameters.drive_lo_frequency = 1.0e9

LF_path = False
start_freq = -50e6
stop_freq = 700e6
measure_q.parameters.resonance_frequency_ge = 0.71e9
# measure_q.parameters.user_defined['amplitude_pi'] = 0.99
# measure_q.parameters.user_defined['amplitude_pi'] = 0.01

### RO spec
# measure_q.parameters.readout_resonator_frequency = 6.816e9
# measure_q.parameters.readout_range_out = -10
# measure_q.parameters.readout_range_in = -15
# measure_q.parameters.user_defined['readout_amp'] = 0.3

# measure_q.parameters.drive_lo_frequency = 7.5e9
# measure_q.parameters.resonance_frequency_ge = 7.5e9

measure_q.parameters.drive_range = -15
# measure_q.parameters.drive_range = 0
# measure_q.parameters.user_defined['reset_length'] = 100e-6
measure_q.parameters.user_defined['reset_length'] = 0.5e-6
# measure_q.parameters.user_defined['pulse_length'] = 1000e-09
measure_q.parameters.user_defined['pulse_length'] = 2000e-09

# measure_q.parameters.user_defined['readout_range_out'] = -10
# measure_q.parameters.user_defined['readout_range_in'] = -10

print(measure_q)

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

#### 3.5.2 Experiment Definition

The frequency sweep of the drive line can now be done in real time (was: near time in older software releases)

In [ ]:
# function that returns a qubit spectroscopy experiment- accepts frequency sweep range as parameter


def qubit_spectroscopy(freq_sweep, drive_pulse, readout_pulse, reset_delay):
    # Create qubit spectroscopy Experiment - uses qubit drive, readout drive and data acquisition lines
    exp_qspec = Experiment(
        uid="Qubit Spectroscopy",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    
    
    # inner loop - real-time averaging - QA in integration mode
    with exp_qspec.acquire_loop_rt(
        uid="freq_shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        
        with exp_qspec.sweep(uid="qfreq_sweep", parameter=freq_sweep):
            # qubit drive
            with exp_qspec.section(uid="qubit_excitation"):
                # exp_qspec.play(signal="drive", pulse=drive_pulse)
                exp_qspec.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
            with exp_qspec.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_qspec.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_qspec.acquire(
                    signal="acquire",
                    handle="qb_spec",
                    kernel=readout_pulse,
                )
            with exp_qspec.section(uid="delay"):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_qspec.delay(signal="measure", time=reset_delay)

    return exp_qspec

In [ ]:
freq_sweep_q = create_drive_freq_sweep(measure_q, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q = Calibration()

if LF_path == True:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
# define experiment with frequency sweep for qubit 0
drive_pulse = create_drive_spec_pulse(measure_q, amp=0.4)
# drive_pulse = create_rabi_drive_pulse(measure_q)

readout_pulse = create_readout_pulse(measure_q)

#update default calibration to qubit settings
device_setup.set_calibration(
    measure_q.calibration()
)

exp_qspec = qubit_spectroscopy(freq_sweep_q, drive_pulse, readout_pulse, reset_delay = measure_q.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec.set_calibration(exp_calibration_q)
exp_qspec.set_signal_map(signal_map_default(measure_q))

In [ ]:
measure_q.parameters.user_defined['reset_length'], freq_sweep_q, drive_pulse, readout_pulse

In [ ]:
# compile the experiment on the open instrument session
compiled_qspec = session.compile(exp_qspec)

#Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)
# generate a pulse sheet to inspect experiment before runtime
#show_pulse_sheet("Pulse_Sheets/Qubit_Spectroscopy", compiled_qspec)

# plot_simulation(compiled_qspec, 0, 10e-6)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")

plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

In [ ]:
measure_q.parameters.resonance_frequency_ge = 0.711e9

## Two-tone vs flux
Fine two tone sweep to find half-flux

In [ ]:
# dc.current_range(1e-3)
start = 6.6e-3
stop = 7.2e-3

current_sweep = np.linspace(start, stop, 41)
dc_q0.output('on')
sweep_qspec_results = []
for current in current_sweep:
    dc_q0.ramp_current(current, 1e-6, 0)
    time.sleep(0.5)
    # run the experiment on qubit 0
    qspec_results = session.run()

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    qspec_res = qspec_results.get_data("qb_spec")
    qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency
    sweep_qspec_results.append(qspec_res)

    # plot measurement data
    # fig = plt.figure()
    # plt.plot(qspec_freq / 1e9, abs(qspec_res), ".k")
    # plt.ylabel("A (a.u.)")
    # plt.xlabel("Frequency (GHz)")

# plt.show()
# dc.ramp_current(5.5e-3,1e-5,0)

In [ ]:
plt.pcolor( current_sweep*1e3,qspec_freq/1e9, np.abs(sweep_qspec_results).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')
# plt.ylim(7.19, 7.215)
# plt.clim(-0.0014, 0.0016)
# plt.axvline(6.2)
# current_sweep[8]*1e3

In [ ]:
angle_data = np.abs(np.unwrap(np.angle(sweep_qspec_results).T))
# for col in angle_data.T:
#     plt.figure()
#     plt.plot(col)

In [ ]:
thresh_low = -2
thresh_high = 2
norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
norm[(norm > thresh_high) | (norm < thresh_low)] = np.nan

In [ ]:
# plt.pcolor( current_sweep*1e3,qspec_freq/1e9, angle_data)
plt.pcolor( current_sweep*1e3,qspec_freq/1e9, norm)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('phase')
# plt.ylim(7.19, 7.215)
# plt.clim(-0.0014, 0.0016)
# plt.axvline(7.95)
# plt.axhline(0.71, color='tab:orange')
# current_sweep[8]*1e3

In [ ]:
for i, col in enumerate(norm.T[:-3]):
    plt.figure()
    plt.plot(qspec_freq/1e9, col)
    plt.xlabel('Frequency [GHz]')
    plt.ylabel('Digitizer voltage (arb.)')
    plt.margins(x=0)
    plt.title(f'Current = {np.round(current_sweep[i]*1e3, 4)} mA')

In [ ]:
peaks_loc = np.nanargmax(norm, axis=0)

In [ ]:
peak_freq = (qspec_freq/1e9)[peaks_loc]

In [ ]:
from scipy import stats
slope, intercept, r_value, p_value, std_err = stats.linregress((current_sweep*1e3)[start:end],peak_freq[start:end])

In [ ]:
slope

In [ ]:
start = -3
end = None
plt.plot((current_sweep*1e3)[start:end], peak_freq[start:end], label='01 transition')
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.title('01 transition vs flux')
plt.margins(x=0, y=0)
# plt.axvline(7.95)
# plt.axhline(0.71, color='tab:orange')

plt.plot((current_sweep*1e3)[start:end], slope*(current_sweep*1e3)[start:end] + intercept, ls='--', alpha=0.4, color='k', label='linear fit')

plt.legend()

# Two-qubit flux xtalk experiment

In [ ]:
from qcodes.instrument_drivers.yokogawa.GS200 import GS200
dc_q1 = GS200('yoko_q1', address = 'GPIB00::15::INSTR')

In [ ]:
from qcodes.instrument_drivers.yokogawa.GS200 import GS200
dc_q0 = GS200('yoko_q0', address = 'TCPIP0::192.168.3.193::inst0::INSTR')

In [ ]:
measure_q = q0

In [ ]:
q0_current_range = [7.05, 7.2]  # mA
q0_frequency_range = [1.2, 1.4]  # GHz

q1_current_range = [5.8, 6.0]  # mA
q1_frequency_range = [1.5, 1.8]  # GHz

In [ ]:
dc_q0.ramp_current(7.2e-3, 1e-6, 0)

## two tone tuneup: q0

In [ ]:
# how many frequency points to measure
qspec_num = 71

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 10

In [ ]:
### QB spec

start_freq = -150e6
stop_freq = 150e6
measure_q.parameters.resonance_frequency_ge = 1.3e9
# measure_q.parameters.user_defined['amplitude_pi'] = 0.99
# measure_q.parameters.user_defined['amplitude_pi'] = 0.01

### RO spec
# measure_q.parameters.readout_resonator_frequency = 6.816e9
# measure_q.parameters.readout_range_out = -10
# measure_q.parameters.readout_range_in = -15
# measure_q.parameters.user_defined['readout_amp'] = 0.3

# measure_q.parameters.drive_lo_frequency = 7.5e9
# measure_q.parameters.resonance_frequency_ge = 7.5e9

measure_q.parameters.drive_range = -15
# measure_q.parameters.drive_range = 0
# measure_q.parameters.user_defined['reset_length'] = 100e-6
measure_q.parameters.user_defined['reset_length'] = 0.5e-6
# measure_q.parameters.user_defined['pulse_length'] = 1000e-09
measure_q.parameters.user_defined['pulse_length'] = 2000e-09

# measure_q.parameters.user_defined['readout_range_out'] = -10
# measure_q.parameters.user_defined['readout_range_in'] = -10

print(measure_q)

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

#### 3.5.2 Experiment Definition

The frequency sweep of the drive line can now be done in real time (was: near time in older software releases)

In [ ]:
# function that returns a qubit spectroscopy experiment- accepts frequency sweep range as parameter


def qubit_spectroscopy(freq_sweep, drive_pulse, readout_pulse, reset_delay):
    # Create qubit spectroscopy Experiment - uses qubit drive, readout drive and data acquisition lines
    exp_qspec = Experiment(
        uid="Qubit Spectroscopy",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    
    
    # inner loop - real-time averaging - QA in integration mode
    with exp_qspec.acquire_loop_rt(
        uid="freq_shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        
        with exp_qspec.sweep(uid="qfreq_sweep", parameter=freq_sweep):
            # qubit drive
            with exp_qspec.section(uid="qubit_excitation"):
                # exp_qspec.play(signal="drive", pulse=drive_pulse)
                exp_qspec.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
            with exp_qspec.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_qspec.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_qspec.acquire(
                    signal="acquire",
                    handle="qb_spec",
                    kernel=readout_pulse,
                )
            with exp_qspec.section(uid="delay"):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_qspec.delay(signal="measure", time=reset_delay)

    return exp_qspec

In [ ]:
freq_sweep_q = create_drive_freq_sweep(measure_q, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q = Calibration()

if LF_path == True:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
# define experiment with frequency sweep for qubit 0
drive_pulse = create_drive_spec_pulse(measure_q, amp=0.4)
# drive_pulse = create_rabi_drive_pulse(measure_q)

readout_pulse = create_readout_pulse(measure_q)

#update default calibration to qubit settings
device_setup.set_calibration(
    measure_q.calibration()
)

exp_qspec_q0 = qubit_spectroscopy(freq_sweep_q, drive_pulse, readout_pulse, reset_delay = measure_q.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q0.set_calibration(exp_calibration_q)
exp_qspec_q0.set_signal_map(signal_map_default(measure_q))

In [ ]:
measure_q.parameters.user_defined['reset_length'], freq_sweep_q, drive_pulse, readout_pulse

In [ ]:
# compile the experiment on the open instrument session
compiled_qspec = session.compile(exp_qspec_q0)

#Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)
# generate a pulse sheet to inspect experiment before runtime
#show_pulse_sheet("Pulse_Sheets/Qubit_Spectroscopy", compiled_qspec)

# plot_simulation(compiled_qspec, 0, 10e-6)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

## two tone tuneup: q1

In [ ]:
measure_q = q1

In [ ]:
dc_q1.ramp_current(5.9e-3, 1e-6, 0)

In [ ]:
# how many frequency points to measure
qspec_num = 271

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 10

In [ ]:
q1?

In [ ]:
### QB spec

start_freq = -400e6
stop_freq = 400e6
measure_q.parameters.resonance_frequency_ge = 1.65e9

### RO spec
# measure_q.parameters.readout_resonator_frequency = 6.816e9
# measure_q.parameters.readout_range_out = -10
# measure_q.parameters.readout_range_in = -15
# measure_q.parameters.user_defined['readout_amp'] = 0.3

# measure_q.parameters.drive_lo_frequency = 7.5e9
# measure_q.parameters.resonance_frequency_ge = 7.5e9

# measure_q.parameters.drive_range = -15
# measure_q.parameters.drive_range = 0
# measure_q.parameters.user_defined['reset_length'] = 100e-6
measure_q.parameters.user_defined['reset_length'] = 0.5e-6
# measure_q.parameters.user_defined['pulse_length'] = 1000e-09
# measure_q.parameters.user_defined['pulse_length'] = 2000e-09

# measure_q.parameters.user_defined['readout_range_out'] = -10
# measure_q.parameters.user_defined['readout_range_in'] = -10

print(measure_q)

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

#### 3.5.2 Experiment Definition

The frequency sweep of the drive line can now be done in real time (was: near time in older software releases)

In [ ]:
num_averages

In [ ]:
num_averages = 13

In [ ]:
# function that returns a qubit spectroscopy experiment- accepts frequency sweep range as parameter


def qubit_spectroscopy(freq_sweep, drive_pulse, readout_pulse, reset_delay):
    # Create qubit spectroscopy Experiment - uses qubit drive, readout drive and data acquisition lines
    exp_qspec = Experiment(
        uid="Qubit Spectroscopy",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    
    
    # inner loop - real-time averaging - QA in integration mode
    with exp_qspec.acquire_loop_rt(
        uid="freq_shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        
        with exp_qspec.sweep(uid="qfreq_sweep", parameter=freq_sweep):
            # qubit drive
            with exp_qspec.section(uid="qubit_excitation"):
                # exp_qspec.play(signal="drive", pulse=drive_pulse)
                exp_qspec.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
            with exp_qspec.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_qspec.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_qspec.acquire(
                    signal="acquire",
                    handle="qb_spec",
                    kernel=readout_pulse,
                )
            with exp_qspec.section(uid="delay"):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_qspec.delay(signal="measure", time=reset_delay)

    return exp_qspec

In [ ]:
freq_sweep_q = create_drive_freq_sweep(measure_q, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q = Calibration()

if LF_path == True:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
# define experiment with frequency sweep for qubit 0
drive_pulse = create_drive_spec_pulse(measure_q, amp=0.4)
# drive_pulse = create_rabi_drive_pulse(measure_q)

readout_pulse = create_readout_pulse(measure_q)

#update default calibration to qubit settings
device_setup.set_calibration(
    measure_q.calibration()
)

exp_qspec_q1 = qubit_spectroscopy(freq_sweep_q, drive_pulse, readout_pulse, reset_delay = measure_q.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q1.set_calibration(exp_calibration_q)
exp_qspec_q1.set_signal_map(signal_map_default(measure_q))

In [ ]:
measure_q.parameters.user_defined['reset_length'], freq_sweep_q, drive_pulse, readout_pulse

In [ ]:
# compile the experiment on the open instrument session
compiled_qspec = session.compile(exp_qspec_q1)

#Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)
# generate a pulse sheet to inspect experiment before runtime
#show_pulse_sheet("Pulse_Sheets/Qubit_Spectroscopy", compiled_qspec)

# plot_simulation(compiled_qspec, 0, 10e-6)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + measure_q.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

## Two-tone vs flux
Fine two tone sweep to find half-flux

In [ ]:
from tqdm import tqdm 
import sys

In [ ]:
q1_set_current = 5.9e-3
dc_q1.ramp_current(q1_set_current, 1e-6, 0)

In [ ]:
start, stop = np.array(q0_current_range)*1e-3
current_sweep = np.linspace(start, stop, 11)

dc_q0.output('on')
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for current in tqdm(current_sweep, file=sys.stdout):
    dc_q0.ramp_current(current, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
def norm(results, thresh_low = -2, thresh_high = 2):
    angle_data = np.abs(np.unwrap(np.angle(results).T))
    norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
    norm[(norm > thresh_high) | (norm < thresh_low)] = np.nan
    return norm

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q0 flux: phase')

plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q0 flux: phase')

In [ ]:
from scipy import stats

def find_peaks(norm, qspec_freq, current_sweep, qubit_label, flux_label, peak_function = np.nanargmin, start=None, end=None):
    peaks_loc = peak_function(norm, axis=0)
    peak_freq = (qspec_freq/1e9)[peaks_loc]    
    slope, intercept, r_value, p_value, std_err = stats.linregress((current_sweep)[start:end],peak_freq[start:end])    

    plt.figure()
    plt.plot((current_sweep)[start:end], peak_freq[start:end], label='01 transition')
    plt.ylabel('frequency (GHz)')
    plt.xlabel('current (mA)')
    plt.title(f'Q{qubit_label} 01 transition vs Q{flux_label} flux')
    plt.margins(x=0)
    plt.plot((current_sweep)[start:end],
             slope*(current_sweep)[start:end] + intercept,
             ls='--',
             alpha=0.4,
             color='k',
             label=f'linear fit: {np.round(slope, 4)} GHz/mA')    
    plt.legend()

    return slope, intercept

In [ ]:
slope_00, offset_00 = find_peaks(norm_0, 
                      qspec_freq_q0, 
                      current_sweep, 
                      qubit_label='0', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

slope_01, offset_01 = find_peaks(norm_1, 
                      qspec_freq_q1, 
                      current_sweep, 
                      qubit_label='1', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

### Tune Q1's flux

In [ ]:
q0_set_current = 7.15e-3
dc_q0.ramp_current(q0_set_current, 1e-6, 0)

In [ ]:
start, stop = np.array(q1_current_range)*1e-3
current_sweep = np.linspace(start, stop, 11)

dc_q1.output('on')
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for current in tqdm(current_sweep, file=sys.stdout):
    dc_q1.ramp_current(current, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q1 flux: phase')

plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q1 flux: phase')

In [ ]:
slope_10, offset_10 = find_peaks(norm_0, 
                      qspec_freq_q0, 
                      current_sweep, 
                      qubit_label='0', 
                      flux_label='0', 
                      peak_function = np.nanargmin,
                      end=6,
                     )

slope_11, offset_11 = find_peaks(norm_1, 
                      qspec_freq_q1, 
                      current_sweep, 
                      qubit_label='1', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

In [ ]:
alpha = 0.7
lw = 3

plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q0/1e9, norm_0)
plt.plot( current_sweep*1e3, slope_10*(current_sweep*1e3) + offset_10, color='tab:orange', alpha=alpha, lw=lw)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('phase')

plt.figure()
plt.pcolor( current_sweep*1e3,qspec_freq_q1/1e9, norm_1)
plt.plot( current_sweep*1e3, slope_11*(current_sweep*1e3) + offset_11, color='tab:orange', alpha=alpha, lw=lw)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('phase')

In [ ]:
flux_xtalk_matrix = np.array([[slope_00, slope_01], [slope_10, slope_11]])

In [ ]:
fig, ax = plt.subplots(figsize=(2,2))
im = ax.imshow(flux_xtalk_matrix, cmap='Blues')

ax.set_xticks(np.arange(flux_xtalk_matrix.shape[0]))
ax.set_yticks(np.arange(flux_xtalk_matrix.shape[1]))

# Loop over data dimensions and create text annotations.
for i in range(flux_xtalk_matrix.shape[0]):
    for j in range(flux_xtalk_matrix.shape[1]):
        v = np.round(flux_xtalk_matrix[i, j], 4)
        if v > -0.5:
            color='w'
        else:
            color='k'
        text = ax.text(j, i, v,
                       ha="center", va="center", color=color)
fig.tight_layout()

In [ ]:
M = flux_xtalk_matrix

In [ ]:
M_inv = np.linalg.inv(M)

In [ ]:
M_inv

In [ ]:
np.dot(M_inv, np.array([1, 1]).T)

In [ ]:
center_current_q0 = 7.12e-3
center_current_q1 = 5.9e-3

In [ ]:
nI = 11
dI = np.linspace(-0.1, 0.1, nI)*1e-3
target_fluxes = np.vstack((dI, np.zeros(nI)))  # small tuning range on q0
dI_c = np.dot(M_inv, target_fluxes)

In [ ]:
center_currents = np.vstack((np.ones(nI)*center_current_q0, np.ones(nI)*center_current_q1))

In [ ]:
current_sweep = dI_c + center_currents

In [ ]:
current_sweep

### Test independent flux control, drive q0 flux

In [ ]:
dc_q0.output('on')
dc_q1.output('on')
dc_q1.ramp_current(center_current_q1, 1e-6, 0)
dc_q0.ramp_current(center_current_q0, 1e-6, 0)

In [ ]:
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for (I0, I1) in tqdm(current_sweep.T, file=sys.stdout):
    dc_q0.ramp_current(I0, 1e-6, 0)
    dc_q1.ramp_current(I1, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
np.abs(np.array(sweep_qspec_results_q0).T).shape

In [ ]:
current_sweep.shape

In [ ]:
plot_curr = (dI + center_current_q0)*1e3

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q0 flux: phase')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q0 flux: phase')

In [ ]:
slope_00, offset_00 = find_peaks(norm_0, 
                      qspec_freq_q0, 
                      plot_curr, 
                      qubit_label='0', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

slope_01, offset_01 = find_peaks(norm_1, 
                      qspec_freq_q1, 
                      plot_curr, 
                      qubit_label='1', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

In [ ]:
print(f'{np.round(np.abs(slope_01/slope_00)*100, 5)} % crosstalk with correction')

In [ ]:
alpha = 0.7
lw = 3

plt.figure()
plt.pcolor( plot_curr,qspec_freq_q0/1e9, norm_0)
plt.plot( plot_curr, slope_00*(plot_curr*1e3) + offset_00, color='tab:orange', alpha=alpha, lw=lw)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q0 flux: phase')

plt.figure()
plt.pcolor( plot_curr,qspec_freq_q1/1e9, norm_1)
plt.plot( plot_curr, slope_01*(plot_curr*1e3) + offset_01, color='tab:orange', alpha=alpha, lw=lw)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q0 flux: phase')

### Test independent flux control, drive q1 flux

In [ ]:
nI = 31
dI = np.linspace(-0.1, 0.1, nI)*1e-3
target_fluxes = np.vstack((np.zeros(nI), dI))  # small tuning range on q1
dI_c = np.dot(M_inv, target_fluxes)

In [ ]:
center_currents = np.vstack((np.ones(nI)*center_current_q0, np.ones(nI)*center_current_q1))

In [ ]:
current_sweep = dI_c + center_currents

In [ ]:
current_sweep

In [ ]:
dc_q0.output('on')
dc_q1.output('on')
dc_q1.ramp_current(center_current_q1, 1e-6, 0)
dc_q0.ramp_current(center_current_q0, 1e-6, 0)

In [ ]:
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for (I0, I1) in tqdm(current_sweep.T, file=sys.stdout):
    dc_q0.ramp_current(I0, 1e-6, 0)
    dc_q1.ramp_current(I1, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
plot_curr = (dI + center_current_q1)*1e3

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q1 flux: phase')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q1 flux: phase')

In [ ]:
slope_10, offset_10 = find_peaks(norm_0, 
                      qspec_freq_q0, 
                      plot_curr, 
                      qubit_label='0', 
                      flux_label='1', 
                      peak_function = np.nanargmin
                     )

slope_11, offset_11 = find_peaks(norm_1, 
                      qspec_freq_q1, 
                      plot_curr, 
                      qubit_label='1', 
                      flux_label='1', 
                      peak_function = np.nanargmin
                     )

In [ ]:
print(f'{np.round(np.abs(slope_10/slope_01)*100, 5)} % crosstalk with correction')

In [ ]:
alpha = 0.7
lw = 3

plt.figure()
plt.pcolor( plot_curr,qspec_freq_q0/1e9, norm_0)
plt.plot( plot_curr, slope_00*(plot_curr*1e3) + offset_00, color='tab:orange', alpha=alpha, lw=lw)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q1 flux: phase')

plt.figure()
plt.pcolor( plot_curr,qspec_freq_q1/1e9, norm_1)
plt.plot( plot_curr, slope_01*(plot_curr*1e3) + offset_01, color='tab:orange', alpha=alpha, lw=lw)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q1 flux: phase')

### Tune q0 to half-flux
lower frequency qubit

#### check that q1 still has small frequency window

In [ ]:
qspec_num = 51
start_freq = -80e6
stop_freq = 80e6
LF_path = False
num_averages=10
q1.parameters.resonance_frequency_ge = 1.88e9

device_setup.set_calibration(
    q1.calibration()
)

In [ ]:
q1.parameters.drive_lo_frequency = q1.parameters.drive_lo_frequency

In [ ]:
freq_sweep_q1 = create_drive_freq_sweep(q1, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q1 = Calibration()

if LF_path == True:
    exp_calibration_q1["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q1,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q1["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q1,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
drive_pulse = create_drive_spec_pulse(q1, amp=0.4)

readout_pulse = create_readout_pulse(q1)

#update default calibration to qubit settings
device_setup.set_calibration(
    q1.calibration()
)

exp_qspec_q1 = qubit_spectroscopy(freq_sweep_q1, drive_pulse, readout_pulse, reset_delay = q1.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q1.set_calibration(exp_calibration_q1)
exp_qspec_q1.set_signal_map(signal_map_default(q1))
compiled_qspec = session.compile(exp_qspec_q1)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

#### adjust two tone window

In [ ]:
dc_q0.ramp_current(8e-3, 1e-6, 0)

In [ ]:
qspec_num = 371
start_freq = -650e6
stop_freq = -500e6
q0.parameters.resonance_frequency_ge = 1.3e9
LF_path = False
num_averages = 11

device_setup.set_calibration(
    q0.calibration()
)

In [ ]:
freq_sweep_q0 = create_drive_freq_sweep(q0, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q0 = Calibration()

if LF_path == True:
    exp_calibration_q0["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q0,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q0["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q0,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
drive_pulse = create_drive_spec_pulse(q0, amp=0.5)

readout_pulse = create_readout_pulse(q0)

#update default calibration to qubit settings
device_setup.set_calibration(
    q0.calibration()
)

exp_qspec_q0 = qubit_spectroscopy(freq_sweep_q0, drive_pulse, readout_pulse, reset_delay = q0.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q0.set_calibration(exp_calibration_q0)
exp_qspec_q0.set_signal_map(signal_map_default(q0))
compiled_qspec = session.compile(exp_qspec_q0)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

#### spectroscopy

In [ ]:
target_cur = 7.986e-3
# target_cur = 7.98e-3
delta_single = target_cur - dc_q0.current()
delta_single

In [ ]:
_delta = np.dot(M_inv, np.array([delta_single, 0]).T)

In [ ]:
center_current_q0 = dc_q0.current() + _delta[0]
center_current_q1 = dc_q1.current() + _delta[1]

In [ ]:
print(center_current_q0, center_current_q1)

In [ ]:
M = [[-1.1013, -0.1293], [-0.1286, -0.9172]]

In [ ]:
M_inv = np.linalg.inv(M)*-1

In [ ]:
nI = 21
dI = np.linspace(-0.2, 0.2, nI)*1e-3
target_fluxes = np.vstack((dI, np.zeros(nI)))  # small tuning range on q0
dI_c = np.dot(M_inv, target_fluxes)

In [ ]:
center_currents = np.vstack((np.ones(nI)*center_current_q0, np.ones(nI)*center_current_q1))

In [ ]:
current_sweep = dI_c + center_currents

In [ ]:
current_sweep

In [ ]:
center_current_q1, center_current_q0

In [ ]:
dc_q0.output('on')
dc_q1.output('on')
dc_q1.ramp_current(center_current_q1, 1e-6, 0)
dc_q0.ramp_current(center_current_q0, 1e-6, 0)

In [ ]:
from tqdm import tqdm

In [ ]:
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for (I0, I1) in tqdm(current_sweep.T, file=sys.stdout):
    dc_q0.ramp_current(I0, 1e-6, 0)
    dc_q1.ramp_current(I1, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
plot_curr = (dI + center_current_q0)*1e3

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
def norm(results, thresh_low = -2, thresh_high = 2):
    angle_data = np.abs(np.unwrap(np.angle(results).T))
    norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
    norm[(norm > thresh_high) | (norm < thresh_low)] = np.nan
    return norm

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q0 flux: phase')
plt.axvline(7.98, ls='--', c='tab:orange')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q0 flux: phase')

In [ ]:
from scipy import stats

def find_peaks_nofit(norm, qspec_freq, current_sweep, qubit_label, flux_label, peak_function = np.nanargmin, start=None, end=None):
    peaks_loc = peak_function(norm, axis=0)
    peak_freq = (qspec_freq/1e9)[peaks_loc]    

    plt.figure()
    plt.plot((current_sweep)[start:end], peak_freq[start:end], label='01 transition')
    plt.ylabel('frequency (GHz)')
    plt.xlabel('current (mA)')
    plt.title(f'Q{qubit_label} 01 transition vs Q{flux_label} flux')
    plt.margins(x=0)

In [ ]:
find_peaks_nofit(norm_0, 
                      qspec_freq_q0, 
                      plot_curr, 
                      qubit_label='0', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

find_peaks_nofit(norm_1, 
                      qspec_freq_q1, 
                      plot_curr, 
                      qubit_label='1', 
                      flux_label='0', 
                      peak_function = np.nanargmin
                     )

### Tune q1 to half-flux
Higher frequency qubit

#### q1 frequency window

In [ ]:
qspec_num = 451
start_freq = -400e6
stop_freq = 630e6
LF_path = False
num_averages=10
q1.parameters.resonance_frequency_ge = 1.3e9
q1.parameters.drive_lo_frequency = 1.3e9

device_setup.set_calibration(
    q1.calibration()
)

In [ ]:
freq_sweep_q1 = create_drive_freq_sweep(q1, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q1 = Calibration()

if LF_path == True:
    exp_calibration_q1["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q1,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q1["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q1,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
drive_pulse = create_drive_spec_pulse(q1, amp=0.4)

readout_pulse = create_readout_pulse(q1)

#update default calibration to qubit settings
device_setup.set_calibration(
    q1.calibration()
)

exp_qspec_q1 = qubit_spectroscopy(freq_sweep_q1, drive_pulse, readout_pulse, reset_delay = q1.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q1.set_calibration(exp_calibration_q1)
exp_qspec_q1.set_signal_map(signal_map_default(q1))
compiled_qspec = session.compile(exp_qspec_q1)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

#### spectroscopy

In [ ]:
_delta = np.dot(M_inv, np.array([-0.1, 0]).T)

In [ ]:
center_current_q0 = 7.12e-3 + _delta[0]*1e-3
center_current_q1 = 5.9e-3 + _delta[1]*1e-3

In [ ]:
M = [[-1.1013, -0.1293], [-0.1286, -0.9172]]

In [ ]:
M_inv = np.linalg.inv(M)*-1

In [ ]:
nI = 31
dI = np.linspace(0, 2, nI)*1e-3
target_fluxes = np.vstack((np.zeros(nI), dI))  # small tuning range on q0
dI_c = np.dot(M_inv, target_fluxes)

In [ ]:
center_currents = np.vstack((np.ones(nI)*center_current_q0, np.ones(nI)*center_current_q1))

In [ ]:
current_sweep = dI_c + center_currents

In [ ]:
current_sweep

In [ ]:
dc_q0.output('on')
dc_q1.output('on')
dc_q1.ramp_current(center_current_q1, 1e-6, 0)
dc_q0.ramp_current(center_current_q0, 1e-6, 0)

In [ ]:
from tqdm import tqdm

In [ ]:
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for (I0, I1) in tqdm(current_sweep.T, file=sys.stdout):
    dc_q0.ramp_current(I0, 1e-6, 0)
    dc_q1.ramp_current(I1, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
plot_curr = (dI + center_current_q1)*1e3

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
def norm(results, thresh_low = -2, thresh_high = 2):
    angle_data = np.abs(np.unwrap(np.angle(results).T))
    norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
    norm[(norm > thresh_high) | (norm < thresh_low)] = np.nan
    return norm

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q1 flux: phase')
# plt.axvline(7.98, ls='--', c='tab:orange')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q1 flux: phase')
plt.axvline(7.1)
plt.axhline(0.96)

In [ ]:
target_cur = 7.20e-3
# target_cur = 7.1e-3
delta_single = target_cur - dc_q1.current()
delta_single

In [ ]:
_delta = np.dot(M_inv, np.array([0, delta_single]).T)

In [ ]:
center_current_q0 = dc_q0.current() + _delta[0]
center_current_q1 = dc_q1.current() + _delta[1]

In [ ]:
print(center_current_q0, center_current_q1)

In [ ]:
dc_q1.ramp_current(center_current_q1, 1e-6, 0)
dc_q0.ramp_current(center_current_q0, 1e-6, 0)

# Coupler g

#### q1 small frequency window

In [ ]:
qspec_num = 151
start_freq = -80e6
stop_freq = 80e6
LF_path = False
q1.parameters.resonance_frequency_ge = 0.96e9

device_setup.set_calibration(
    q1.calibration()
)

In [ ]:
freq_sweep_q1 = create_drive_freq_sweep(q1, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q1 = Calibration()

if LF_path == True:
    exp_calibration_q1["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q1,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q1["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q1,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
drive_pulse = create_drive_spec_pulse(q1, amp=0.4)

readout_pulse = create_readout_pulse(q1)

#update default calibration to qubit settings
device_setup.set_calibration(
    q1.calibration()
)

exp_qspec_q1 = qubit_spectroscopy(freq_sweep_q1, drive_pulse, readout_pulse, reset_delay = q1.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q1.set_calibration(exp_calibration_q1)
exp_qspec_q1.set_signal_map(signal_map_default(q1))
compiled_qspec = session.compile(exp_qspec_q1)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

#### adjust two tone window

In [ ]:
nI = 3
dI = np.linspace(8.3e-3 - center_current_q0, 8.5e-3 - center_current_q0, nI)
target_fluxes = np.vstack((dI, np.zeros(nI)))  # small tuning range on q0
dI_c = np.dot(M_inv, target_fluxes)

In [ ]:
center_currents = np.vstack((np.ones(nI)*center_current_q0, np.ones(nI)*center_current_q1))

In [ ]:
current_sweep = dI_c + center_currents

In [ ]:
current_sweep

In [ ]:
dc_q0.ramp_current(0.00826331, 1e-6, 0)
dc_q1.ramp_current(0.00705209, 1e-6, 0)

In [ ]:
qspec_num = 471
start_freq = 200e6
stop_freq = 350e6
q0.parameters.resonance_frequency_ge = 0.71e9
LF_path = False

device_setup.set_calibration(
    q0.calibration()
)

In [ ]:
freq_sweep_q0 = create_drive_freq_sweep(q0, start_freq, stop_freq, qspec_num)

# experiment signal calibration for qubit, specifically overwriting the frequency to the sweep parameter
exp_calibration_q0 = Calibration()

if LF_path == True:
    exp_calibration_q0["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q0,
            modulation_type=ModulationType.HARDWARE,
        ),
        port_mode = PortMode.LF
    )
else:
    exp_calibration_q0["drive"] = SignalCalibration(
        oscillator=Oscillator(
            frequency=freq_sweep_q0,
            modulation_type=ModulationType.HARDWARE,
        ),
    )

#### 3.5.3 Run and Evaluate Experiment for Both Qubits

Runs the experiment and evaluates the data returned by the measurement

In [ ]:
drive_pulse = create_drive_spec_pulse(q0, amp=0.3)

readout_pulse = create_readout_pulse(q0)

#update default calibration to qubit settings
device_setup.set_calibration(
    q0.calibration()
)

exp_qspec_q0 = qubit_spectroscopy(freq_sweep_q0, drive_pulse, readout_pulse, reset_delay = q0.parameters.user_defined['reset_length'])

# apply calibration and signal map for qubit for this experiment
exp_qspec_q0.set_calibration(exp_calibration_q0)
exp_qspec_q0.set_signal_map(signal_map_default(q0))
compiled_qspec = session.compile(exp_qspec_q0)

In [ ]:
# run the experiment on qubit 0
qspec_results = session.run()

timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
print(f"File saved as Results_Needed/qspec_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
qspec_res = qspec_results.get_data("qb_spec")
qspec_freq = qspec_results.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency

# plot measurement data
fig = plt.figure()
plt.plot(qspec_freq / 1e9, np.angle(qspec_res), ".k", ls='-')
plt.ylabel("A (a.u.)")
plt.xlabel("Frequency (GHz)")
plt.margins(x=0)

# plt.axvline(0.711, linestyle = '--', alpha=0.4)
# plt.axvline(qspec_freq[np.argmin(abs(qspec_res))]/1e9 , linestyle = '--')
print('estimated frequency ' + str(qspec_freq[np.argmin(np.angle(qspec_res))]/1e9) + ' GHz')
plt.show()

### q0 resonator spectroscopy check

In [ ]:
# frequency range of spectroscopy scan -
# around expected resonator frequency as defined in qubit parameters
start_freq = -20e6
stop_freq = 15e6
num_points = 71

# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 10

readout_pulse = create_readout_pulse(q0)

In [ ]:
# function that defines a resonator spectroscopy experiment, and takes the frequency sweep as a parameter

def res_spectroscopy_pulsed(freq_sweep, num_averages, readout_pulse):
    # Create resonator spectroscopy experiment - uses only readout drive and signal acquisition
    exp_spec_pulsed = Experiment(
        uid="Resonator Spectroscopy",
        signals=[
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define experimental sequence
    # outer loop - vary drive frequency

    # inner loop - average multiple measurements for each frequency - measurement in spectroscopy mode
    with exp_spec_pulsed.acquire_loop_rt(
        uid="shots",
        count=2**num_averages,
        acquisition_type=AcquisitionType.SPECTROSCOPY,
        averaging_mode=AveragingMode.SEQUENTIAL,
    ):
        with exp_spec_pulsed.sweep(
            uid="res_freq",
            parameter=freq_sweep,
            chunk_count=1,
        ):
            # readout pulse and data acquisition
            with exp_spec_pulsed.section(uid="spectroscopy"):
                # play resonator excitation pulse
                exp_spec_pulsed.play(signal="measure", pulse=readout_pulse)
                # resonator signal readout
                exp_spec_pulsed.acquire(
                    signal="acquire", handle="res_spec_pulsed", length=readout_pulse.length
                )
            with exp_spec_pulsed.section(uid="delay", length=10e-6):
                # holdoff time after signal acquisition - minimum 1us required for data processing on UHFQA
                exp_spec_pulsed.reserve(signal="measure")

    return exp_spec_pulsed

In [ ]:
# measure_q.parameters.readout_range_out = 5

# # apply calibration to device setup
device_setup.set_calibration(
    q0.calibration()
)

In [ ]:
# create freq sweep
freq_sweep = create_readout_freq_sweep(q0, start_freq, stop_freq, num_points)

# define the experiment with the frequency sweep relevant for qubit
exp_spec_pulsed = res_spectroscopy_pulsed(freq_sweep, num_averages, readout_pulse)

# set signal calibration and signal map for experiment to qubit
exp_spec_pulsed.set_calibration(res_spec_calib(freq_sweep))
exp_spec_pulsed.set_signal_map(res_spec_map(q0))

In [ ]:
# compile the experiment on the open instrument session
compiled_spec_pulsed = session.compile(exp_spec_pulsed)

Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)

# generate a pulse sheet to inspect experiment before runtime
show_pulse_sheet("Pulse_Sheets/Pulsed_Spectroscopy", compiled_spec_pulsed)

In [ ]:
# run the experiment on the open instrument session
spec_pulsed_results = session.run()

# save the data
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/spec_pulsed_results_{timestamp}.json")
print(f"File saved as Results_Needed/spec_pulsed_results_{timestamp}.json")

In [ ]:
electrical_delay = 60.68e-9
# electrical_delay = 1
# _delay = 10e9

# get the measurement data returned by the instruments from the QCCS session
spec_res = spec_pulsed_results.get_data("res_spec_pulsed")
# define the frequency axis from the qubit parameters
spec_freq = q0.parameters.readout_lo_frequency + spec_pulsed_results.get_axis("res_spec_pulsed")[0]
# %matplotlib qt
if emulate:
    # create some dummy data if running in emulation mode
    spec_res = lorentzian(
        spec_freq,
        10e6,
        q0.parameters.readout_frequency * (0.995 + 0.01 * np.random.rand(1)[0])
        + q0.parameters.readout_lo_frequency,
        -1e7,
        10,
    ) + 0.2 * np.random.rand(len(spec_freq))

# plot the measurement data
fig, [ax1, ax2, ax3] = plt.subplots(3, 1, figsize=(6, 7))
ax1.plot(spec_freq / 1e9, abs(spec_res), ".k")

ax2.plot(spec_freq / 1e9, np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res), "orange")
ax3.plot(spec_freq / 1e9, np.unwrap(np.angle(np.exp(1j*electrical_delay*2*np.pi*spec_freq)*spec_res)), "orange")

ax1.set_ylabel("A (a.u.)")
# ax1.set_title('Mag')

ax2.set_ylabel("$\\phi$ (rad)")
# ax2.set_title('Wrapped phase')

ax3.set_ylabel("$\\phi$ (rad)")
# ax3.set_title('Unwrapped phase')
ax3.set_xlabel("Frequency (GHz)")

vl = [6.817]
for _vl in vl:
    ax1.axvline(_vl, ls='--', 
                color='tab:purple',
                alpha=0.4,
               )
    ax2.axvline(_vl, ls='--', 
                color='tab:purple',
                alpha=0.4,
               )
    ax3.axvline(_vl, ls='--', 
                color='tab:purple',
                alpha=0.4,
               )

plt.tight_layout()

# plt.axvline(q0.parameters.readout_resonator_frequency*1e-9)

In [ ]:
q0.parameters.readout_resonator_frequency = 6.817e9

#### spectroscopy

In [ ]:
target_cur = 7.986e-3
# target_cur = 7.98e-3
delta_single = target_cur - dc_q0.current()
delta_single

In [ ]:
_delta = np.dot(M_inv, np.array([delta_single, 0]).T)

In [ ]:
center_current_q0 = dc_q0.current() + _delta[0]
center_current_q1 = dc_q1.current() + _delta[1]

In [ ]:
print(center_current_q0, center_current_q1)

In [ ]:
M = [[-1.1013, -0.1293], [-0.1286, -0.9172]]

In [ ]:
M_inv = np.linalg.inv(M)*-1

In [ ]:
# (0.007822207359938563, 0.007113941488783145)
center_current_q0, center_current_q1

In [ ]:
8.45e-3 - center_current_q0

In [ ]:
nI = 41
# dI = np.linspace(8.15, 1.3, nI)*1e-3
dI = np.linspace(8.15e-3 - center_current_q0, 8.5e-3 - center_current_q0, nI)
target_fluxes = np.vstack((dI, np.zeros(nI)))  # small tuning range on q0
dI_c = np.dot(M_inv, target_fluxes)

In [ ]:
center_currents = np.vstack((np.ones(nI)*center_current_q0, np.ones(nI)*center_current_q1))

In [ ]:
current_sweep = dI_c + center_currents

In [ ]:
dc_q1.ramp_current(center_current_q1, 1e-6, 0)
dc_q0.ramp_current(center_current_q0, 1e-6, 0)

In [ ]:
from tqdm import tqdm

In [ ]:
sweep_qspec_results_q0 = []
sweep_qspec_results_q1 = []
for (I0, I1) in tqdm(current_sweep.T, file=sys.stdout):
    dc_q0.ramp_current(I0, 1e-6, 0)
    dc_q1.ramp_current(I1, 1e-6, 0)
    time.sleep(0.5)

    timestamp = time.strftime("%Y%m%dT%H%M%S")
    Path("Results_Needed").mkdir(parents=True, exist_ok=True)
    
    compiled_qspec = session.compile(exp_qspec_q0)
    qspec_results_q0 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")

    compiled_qspec = session.compile(exp_qspec_q1)
    qspec_results_q1 = session.run()
    session.save_results(f"Results_Needed/qspec_results_{timestamp}.json")
    
    # get measurement data returned by the instruments
    
    qspec_res_q0 = qspec_results_q0.get_data("qb_spec")
    qspec_freq_q0 = qspec_results_q0.get_axis("qb_spec")[0] + q0.parameters.drive_lo_frequency
    sweep_qspec_results_q0.append(qspec_res_q0)
    
    
    qspec_res_q1 = qspec_results_q1.get_data("qb_spec")
    qspec_freq_q1 = qspec_results_q1.get_axis("qb_spec")[0] + q1.parameters.drive_lo_frequency
    sweep_qspec_results_q1.append(qspec_res_q1)

In [ ]:
plot_curr = (dI + center_current_q0)*1e3

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, np.abs(sweep_qspec_results_q0).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, np.abs(sweep_qspec_results_q1).T)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('mag')

In [ ]:
def norm(results, thresh_low = -2, thresh_high = 2):
    angle_data = np.abs(np.unwrap(np.angle(results).T))
    norm = np.array(angle_data) - np.nanmedian(angle_data, axis=0)
    norm[(norm > thresh_high) | (norm < thresh_low)] = np.nan
    return norm

In [ ]:
norm_0 = norm(sweep_qspec_results_q0)
norm_1 = norm(sweep_qspec_results_q1)

In [ ]:
plt.figure()
plt.pcolor(plot_curr,qspec_freq_q0/1e9, norm_0)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q0 01 vs Q0 flux: phase')
# plt.xlim(8.2, 8.45)
# plt.ylim(0.9, 1.05)
# plt.axvline(8.2, ls='--', c='tab:orange')
# plt.axvline(8.45, ls='--', c='tab:orange')
plt.axhline(0.983, ls='--', c='tab:orange', alpha=0.5)
plt.axhline(0.968, ls='--', c='tab:orange', alpha=0.5)

plt.figure()
plt.pcolor(plot_curr,qspec_freq_q1/1e9, norm_1)
plt.ylabel('frequency (GHz)')
plt.xlabel('current (mA)')
plt.colorbar()
plt.title('Q1 01 vs Q0 flux: phase')

In [ ]:
dc_q0.ramp_current(0, 1e-6, 0)
dc_q1.ramp_current(0, 1e-6, 0)

### 3.6 Amplitude Rabi Experiment

Sweep the pulse amplitude of a qubit drive pulse to determine the ideal amplitudes for specific qubit rotation angles

#### 3.6.1 Additional Experimental Parameters

Define the amplitude sweep range and qubit excitation pulse

In [ ]:
dc_q0.ramp_current(7.95e-3, 1e-6, 0)

In [ ]:
measure_q.parameters.resonance_frequency_ge = 0.710e9

In [ ]:
measure_q.parameters.user_defined['reset_length'] = 100e-6
# measure_q.parameters.user_defined['reset_length'] = e-6

measure_q.parameters.user_defined['pulse_length'] = 2000e-9
# measure_q.parameters.user_defined['pulse_length'] = 5e-6

measure_q.parameters.user_defined['amplitude_pi'] = 0.99
measure_q.parameters.drive_range = -15
print(measure_q)

In [ ]:
# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 14

# define number of amplitude points, from 0 to 1 in gain
num_amplitudes = 41
amp_min=0.01
amp_max=0.5
# amp_max=0.


#### 3.6.2 Experiment Definition

Define the experimental pulse and readout sequence - here without any explicit qubit reference

Explicit qubit reference is then given through different experimental calibration and signal maps

In [ ]:
def create_square_pulse(qubit):
    pulse = pulse_library.const(
        uid=f"square_pulse_{qubit.uid}",
        length=qubit.parameters.user_defined["pulse_length"],
        amplitude=1, #max power to start
        # amplitude=amp, #max power to start
    )

    return pulse

In [ ]:
# function that returns an amplitude Rabi experiment


def amplitude_rabi(drive_pulse, readout_pulse, amplitude_sweep, relax_time = 1e-6):
    exp_rabi = Experiment(
        uid="Amplitude Rabi",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define Rabi experiment pulse sequence
    # outer loop - real-time, cyclic averaging
    with exp_rabi.acquire_loop_rt(
        uid="rabi_shots",
        count=2**num_averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        
        # inner loop - real time sweep of Rabi ampitudes
        with exp_rabi.sweep(uid="rabi_sweep", parameter=amplitude_sweep):
            # play qubit excitation pulse - pulse amplitude is swept
            with exp_rabi.section(
                uid="qubit_excitation", alignment=SectionAlignment.RIGHT
            ):
                exp_rabi.play(
                    signal="drive", pulse=drive_pulse, amplitude=amplitude_sweep, marker = {"marker1": {"enable": True}}
                )
            # readout pulse and data acquisition
            with exp_rabi.section(uid="readout_section", play_after="qubit_excitation"):
                # play readout pulse on measure line
                exp_rabi.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_rabi.acquire(
                    signal="acquire",
                    handle="amp_rabi",
                    kernel=readout_pulse,
                )
                
            with exp_rabi.section(uid="delay", length=relax_time):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_rabi.reserve(signal="measure")
    return exp_rabi

#### 3.6.3 Set Experiment Parameters and Compile

In [ ]:
def create_rabi_amp_sweep(qubit, amp_num, amp_min=0.1, amp_max=0.22, uid="rabi_amp"):
    return LinearSweepParameter(uid=uid, start=amp_min, stop=amp_max, count=amp_num)

In [ ]:
# set signal map for qubit - no experimental calibration necessary, calibration taken from DeviceSetup, i.e. baseline
device_setup.set_calibration(
    measure_q.calibration()
)

drive_pulse = create_rabi_drive_pulse(measure_q)
# drive_pulse = create_drive_spec_pulse(measure_q)
# drive_pulse = create_square_pulse(measure_q)

exp_rabi = amplitude_rabi(
    drive_pulse, readout_pulse, create_rabi_amp_sweep(qubit=measure_q, amp_num=num_amplitudes, amp_min=amp_min, amp_max=amp_max), relax_time = measure_q.parameters.user_defined['reset_length']
)

exp_rabi.set_signal_map(signal_map_default(measure_q))

# compile the experiment on the open instrument session
compiled_rabi = session.compile(exp_rabi)

#### 3.6.4 Show Pulse Sheet

In [ ]:
show_pulse_sheet("Pulse_Sheets/Amplitude_Rabi", compiled_rabi)

#### 3.6.5 Plot Simulated Outputs

In [ ]:
# Simulate experiment
# plot_simulation(compiled_rabi, 0e-6, 30e-6)

#### 3.6.6 Run, Save, and Plot Results

Finally, you'll run the experiment, save, and plot the results.

In [ ]:
# run the compiled experiemnt
rabi_results = session.run(compiled_rabi)
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/rabi_results_{timestamp}.json")
print(f"File saved as Results_Needed/rabi_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
rabi_res = rabi_results.get_data("amp_rabi")

# define amplitude axis from qubit parameters
rabi_amp = rabi_results.get_axis("amp_rabi")[0]

In [ ]:
# plot measurement data
fig = plt.figure()
plt.plot(rabi_amp, np.angle(rabi_res), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("amplitude (a.u.)")

# increase number of plot points for smooth plotting of fit results
amp_plot = np.linspace(rabi_amp[0], rabi_amp[-1], 5 * len(rabi_amp))

# fit measurement results - assume sinusoidal oscillation with drive amplitude
# popt, pcov = oscillatory.fit(rabi_amp, rabi_res, 10, 0, 1, 1.2, plot=False)
# print(f"Fitted parameters: {popt}")

# plot fit results together with measurement data
# plt.plot(amp_plot, oscillatory(amp_plot, *popt), "-r");
# plt.axvline(0.22)
# pi_amp = (np.pi-(np.pi-popt[1]))/(popt[0])
# plt.axvline(0.5, alpha=0.5)
plt.axvline(0.39, alpha=0.5)
# pi_amp = 0.95

plt.title('Resonator phase response')

In [ ]:
# plot measurement data
fig = plt.figure()
plt.plot(rabi_amp, rabi_res, ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("amplitude (a.u.)")

# increase number of plot points for smooth plotting of fit results
amp_plot = np.linspace(rabi_amp[0], rabi_amp[-1], 5 * len(rabi_amp))

# fit measurement results - assume sinusoidal oscillation with drive amplitude
# popt, pcov = oscillatory.fit(rabi_amp, rabi_res, 10, 0, 1, 1.2, plot=False)
# print(f"Fitted parameters: {popt}")

# plot fit results together with measurement data
# plt.plot(amp_plot, oscillatory(amp_plot, *popt), "-r");
# plt.axvline(0.22)
# pi_amp = (np.pi-(np.pi-popt[1]))/(popt[0])
# plt.axvline(0.51, alpha=0.5)
# plt.axvline(0.28, alpha=0.5)
# pi_amp = 0.95

plt.title('Resonator mag response')

In [ ]:
print(measure_q)

In [ ]:
# rabi_amplitude = pi_amp
measure_q.parameters.user_defined['amplitude_pi'] = 0.39
# measure_q.parameters.user_defined['amplitude_pi/2'] = 0.51

# T1

#### 3.7.1 Experiment Parameters

In [ ]:
measure_q?

In [ ]:
measure_q.parameters.user_defined['reset_length'] = 150e-6
# measure_q.parameters.user_defined['pulse_length'] = 20e-6
# measure_q.parameters.drive_range = 10

In [ ]:
# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 13

# define delay sweep
n_steps = 25
start_delay = 0e-6
stop_delay = 200e-6


#### 3.7.2 Experiment Definition

In [ ]:
# function that returns a T1 experiment


def T1(drive_pulse, readout_pulse, time_sweep, relax_time = 5e-6):
    exp_T1 = Experiment(
        uid="T1 Experiment",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define T1 experiment pulse sequence
    # outer loop - real-time, cyclic averaging
    with exp_T1.acquire_loop_rt(
        uid="T1_shots",
        count=2**num_averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
        #repetition_mode=RepetitionMode.AUTO,
    ):
        
        # inner loop - real time sweep of T1 time delays
        with exp_T1.sweep(
            uid="T1_sweep", parameter=time_sweep, alignment=SectionAlignment.RIGHT
        ):
            # play qubit excitation pulse - delay is swept
            with exp_T1.section(
                uid="qubit_excitation", alignment=SectionAlignment.RIGHT
            ):
                exp_T1.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
                exp_T1.delay(signal="drive", time=time_sweep)
            # readout pulse and data acquisition
            with exp_T1.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_T1.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_T1.acquire(
                    signal="acquire",
                    handle="T1",
                    kernel=readout_pulse,
                )
            with exp_T1.section(uid="delay", length=relax_time):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_T1.reserve(signal="measure")
    return exp_T1

#### 3.7.3 Create Experiment and Signal Map

In [ ]:
# define pulses and create experiment
readout_pulse = create_readout_pulse(measure_q)
drive_pulse = create_T1_drive_pulse(measure_q)
time_sweep = create_delay_sweep(start=start_delay, stop=stop_delay, count=n_steps)

#update calibration to default
device_setup.set_calibration(
    measure_q.calibration()
)

T1_exp = T1(
    drive_pulse=drive_pulse, readout_pulse=readout_pulse, time_sweep=time_sweep, relax_time = measure_q.parameters.user_defined['reset_length']
)

T1_exp.set_signal_map(signal_map_default(measure_q))

compiled_T1 = session.compile(T1_exp)

#### 3.7.4 Show Pulse Sheet

In [ ]:
show_pulse_sheet("Pulse_Sheets/T1", compiled_T1)

#### 3.7.5 Plot Simulated Outputs

In [ ]:
# plot_simulation(compiled_T1, 100e-6, 150e-6, plot_width=10)

#### 3.7.6 Run, Save, and Plot Results

In [ ]:
# run the compiled experiemnt
T1_results = session.run()
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results_Needed").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results_Needed/T1_results_{timestamp}.json")
print(f"File saved as Results_Needed/T1_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
T1_res = T1_results.get_data("T1")

# define time axis from qubit parameters
T1_delay = T1_results.get_axis("T1")[0]

if emulate:
    # create dummy data if running in emulation mode
    T1_res = exponential_decay(
        T1_delay, 2e5, amplitude=0.5, offset=0.5
    ) + 0.12 * np.random.rand(len(T1_delay))

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(T1_delay*1e6, np.angle(T1_res), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (us)")

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(T1_delay*1e6, T1_res, ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (us)")

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(T1_delay, np.angle(T1_res), ".k")
plt.ylabel("A (a.u.)")
plt.xlabel("delay (us)")

# increase number of plot points for smooth plotting of fit results
#delay_plot = np.linspace(T1_delay[0], T1_delay[-1], 5 * len(T1_delay))

## fit measurement data to decaying sinusoidal oscillatio
popt, pcov = exponential_decay.fit(
   T1_delay,
   np.angle(T1_res),
   1/3.4e-6,
   -20,
   -20,
   plot=True,
   # bounds=[
   #     [0.01e6, -np.pi / 2, 0.1 / 1 / 10e-6, 0.2, 0.2],
   #     [15e6, np.pi / 2, 10 / 1 / 10e-6, 2, 2],
   # ],
)
print(f"Fitted parameters: {popt}")
print('T1 time ' + str(1/popt[0]*1e6) + ' us') 
# plot fit results together with experimental data
#plt.plot(delay_plot, oscillatory_decay(delay_plot, *popt), "-r");

In [ ]:
print(measure_q)

### 3.7 Ramsey Experiment
The Ramsey experiment is different from the experiments above as the length of the drive section changes. Using a right-aligned sweep section and the automatic repetition time makes sure that the experiment is run as efficiently as possible on the Zurich Instruments hardware.

#### 3.7.1 Experiment Parameters

In [ ]:
# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 12

# define delay sweep
n_steps = 41
start_delay = 0e-6
stop_delay = 20e-6


#### 3.7.2 Experiment Definition

In [ ]:
def create_ramsey_drive_pulse(qubit):
    return pulse_library.gaussian(
        uid=f"gaussian_drive_{qubit.uid}",
        length=qubit.parameters.user_defined['pulse_length'],
        amplitude=qubit.parameters.user_defined['amplitude_pi']/2,
    )

In [ ]:
# function that returns a Ramsey experiment


def ramsey(drive_pulse, readout_pulse, time_sweep, relax_time = 5e-6):
    exp_ramsey = Experiment(
        uid="Ramsey Experiment",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define Ramsey experiment pulse sequence
    # outer loop - real-time, cyclic averaging
    with exp_ramsey.acquire_loop_rt(
        uid="ramsey_shots",
        count=2**num_averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
        #repetition_mode=RepetitionMode.AUTO,
    ):
            
        # inner loop - real time sweep of Ramsey time delays
        with exp_ramsey.sweep(
            uid="ramsey_sweep", parameter=time_sweep, alignment=SectionAlignment.RIGHT
        ):
            # play qubit excitation pulse - pulse delay is swept
            with exp_ramsey.section(
                uid="qubit_excitation", alignment=SectionAlignment.RIGHT
            ):
                exp_ramsey.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
                exp_ramsey.delay(signal="drive", time=time_sweep)
                exp_ramsey.play(signal="drive", pulse=drive_pulse, marker = {"marker1": {"enable": True}})
            # readout pulse and data acquisition
            with exp_ramsey.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_ramsey.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_ramsey.acquire(
                    signal="acquire",
                    handle="ramsey",
                    kernel=readout_pulse,
                )
            with exp_ramsey.section(uid="delay", length=relax_time):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_ramsey.reserve(signal="measure")
    return exp_ramsey

In [ ]:
# #measure_q.parameters.user_defined['reset_length'] = 1e-3
# measure_q.parameters.resonance_frequency_ge = measure_q.parameters.resonance_frequency_ge
# print(measure_q)

#### 3.7.3 Create Experiment and Signal Map

In [ ]:
# define pulses and create experiment
readout_pulse = create_readout_pulse(measure_q)
drive_pulse = create_ramsey_drive_pulse(measure_q)
time_sweep = create_delay_sweep(start=start_delay, stop=stop_delay, count=n_steps)

#update calibration to default
device_setup.set_calibration(
    measure_q.calibration()
)

ramsey_exp = ramsey(
    drive_pulse=drive_pulse, readout_pulse=readout_pulse, time_sweep=time_sweep, relax_time = measure_q.parameters.user_defined['reset_length']
)

ramsey_exp.set_signal_map(signal_map_default(measure_q))

compiled_ramsey = session.compile(ramsey_exp)

#### 3.7.4 Show Pulse Sheet

In [ ]:
show_pulse_sheet("Pulse_Sheets/Ramsey", compiled_ramsey)

#### 3.7.5 Plot Simulated Outputs

In [ ]:
# plot_simulation(compiled_ramsey, 50e-6, 100e-6, plot_width=10)

#### 3.7.6 Run, Save, and Plot Results

In [ ]:
# run the compiled experiemnt
ramsey_results = session.run()
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results/ramsey_results_{timestamp}_.json")
print(f"File saved as Results/ramsey_results_{timestamp}_.json")

In [ ]:
# get measurement data returned by the instruments
ramsey_res = ramsey_results.get_data("ramsey")

# define time axis from qubit parameters
ramsey_delay = ramsey_results.get_axis("ramsey")[0]

if emulate:
    # create dummy data if running in emulation mode
    ramsey_res = oscillatory_decay(
        ramsey_delay, 1e6, 0, 1 / 10e-6, amplitude=0.5, offset=0.5
    ) + 0.12 * np.random.rand(len(ramsey_delay))

In [ ]:
# plot measurement results
fig = plt.figure()
plt.plot(ramsey_delay, np.angle(ramsey_res), ".k")
plt.plot(ramsey_delay, np.angle(ramsey_res), linestyle ='--')

plt.ylabel("A (a.u.)")
plt.xlabel("delay (s)")

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(ramsey_delay[0], ramsey_delay[-1], 5 * len(ramsey_delay))

## fit measurement data to decaying sinusoidal oscillatio
popt, pcov = oscillatory_decay.fit(
    ramsey_delay,
    np.angle(ramsey_res),
    10/0.5e-6,
    0,
    1 / 4e-6,
    0.002,
    0.0008,
    plot=True,
    # bounds=[
    #     [0.01e6, -np.pi / 2, 0.1 / 1 / 10e-6, 0.2, 0.2],
    #     [15e6, np.pi / 2, 10 / 1 / 10e-6, 2, 2],
    # ],
)
print(f"Fitted parameters: {popt}")

In [ ]:
print(f'T2r = {1e6/popt[2]} us')

In [ ]:
# measure_q.parameters.resonance_frequency_ge = measure_q.parameters.resonance_frequency_ge + 0.5e6

In [ ]:
measure_q.parameters.resonance_frequency_ge

In [ ]:
measure_q.parameters.resonance_frequency_ge = 1.1607e9

In [ ]:
device_setup.set_calibration(
    measure_q.calibration()
)

### 3.9 echo Experiment
Adding a Y180 pulse in middle of Ramsey

In [ ]:
print(measure_q)

In [ ]:
measure_q.parameters.user_defined['reset_length'] = 300e-6

In [ ]:
# define number of averages
# used for 2^num_averages, maximum: num_averages = 17
num_averages = 14

# define delay sweep
n_steps = 41
start_delay = 0e-6
stop_delay = 80e-6


#### 3.9.2 Experiment Definition

In [ ]:
# function that returns an Echo experiment


def echo(x90_pulse, y180_pulse, readout_pulse, time_sweep, relax_time = 5e-6):
    exp_echo = Experiment(
        uid="Echo Experiment",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define Echo experiment pulse sequence
    # outer loop - real-time, cyclic averaging
    with exp_echo.acquire_loop_rt(
        uid="echo_shots",
        count=2**num_averages,
        averaging_mode=AveragingMode.CYCLIC,
        acquisition_type=AcquisitionType.INTEGRATION,
        #repetition_mode=RepetitionMode.AUTO,
    ):
            
        # inner loop - real time sweep of echo time delays
        with exp_echo.sweep(
            uid="echo_sweep", parameter=time_sweep, alignment=SectionAlignment.RIGHT
        ):
            # play qubit excitation pulse
            with exp_echo.section(
                uid="qubit_excitation", alignment=SectionAlignment.RIGHT
            ):
                exp_echo.play(signal="drive", pulse=x90_pulse, phase = 0, marker = {"marker1": {"enable": True}})
                exp_echo.delay(signal="drive", time=time_sweep/2)
                exp_echo.play(signal="drive", pulse=y180_pulse, phase = 0, marker = {"marker1": {"enable": True}})
                exp_echo.delay(signal="drive", time=time_sweep/2)
                exp_echo.play(signal="drive", pulse=x90_pulse, phase = 0, marker = {"marker1": {"enable": True}})
            # readout pulse and data acquisition
            with exp_echo.section(
                uid="readout_section", play_after="qubit_excitation"
            ):
                # play readout pulse on measure line
                exp_echo.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_echo.acquire(
                    signal="acquire",
                    handle="echo",
                    kernel=readout_pulse,
                )
            with exp_echo.section(uid="delay", length=relax_time):
                # relax time after readout - for qubit relaxation to groundstate and signal processing
                exp_echo.reserve(signal="measure")
    return exp_echo

In [ ]:
# measure_q.parameters.resonance_frequency_ge = 4.5509e9+190e3
# measure_q.parameters.user_defined['reset_length'] = 5e-3
print(measure_q)

#### 3.9.3 Create Experiment and Signal Map

In [ ]:
# define pulses and create experiment
readout_pulse = create_readout_pulse(measure_q)
x90_pulse = create_pi_2_pulse(measure_q)
y180_pulse = create_pi_pulse(measure_q)
time_sweep = create_delay_sweep(start=start_delay, stop=stop_delay, count=n_steps)

#update calibration to default
device_setup.set_calibration(
    measure_q.calibration()
)

echo_exp = echo(
    x90_pulse=x90_pulse, y180_pulse=y180_pulse, readout_pulse=readout_pulse, time_sweep=time_sweep, relax_time = measure_q.parameters.user_defined['reset_length']
)

echo_exp.set_signal_map(signal_map_default(measure_q))

compiled_echo = session.compile(echo_exp)

#### 3.9.4 Show Pulse Sheet

In [ ]:
show_pulse_sheet("Pulse_Sheets/echo", compiled_echo)

#### 3.9.5 Plot Simulated Outputs

In [ ]:
# plot_simulation(compiled_echo, 4000e-6, 4050e-6, plot_width=10)

#### 3.9.6 Run, Save, and Plot Results

In [ ]:
# run the compiled experiemnt
echo_results = session.run()
timestamp = time.strftime("%Y%m%dT%H%M%S")
Path("Results").mkdir(parents=True, exist_ok=True)
session.save_results(f"Results/echo_results_{timestamp}.json")
print(f"File saved as Results/echo_results_{timestamp}.json")

In [ ]:
# get measurement data returned by the instruments
echo_res = echo_results.get_data("echo")

# define time axis from qubit parameters
echo_delay = echo_results.get_axis("echo")[0]


In [ ]:
plt.plot(echo_delay, np.angle(echo_res))

In [ ]:
plt.plot(echo_delay, np.angle(echo_res), ls='-', marker='.', color='k')
popt, pcov = exponential_decay.fit(
   echo_delay,
   np.angle(echo_res),
   1/3.4e-6,
   -20,
   -20,
   plot=True,
   # bounds=[
   #     [0.01e6, -np.pi / 2, 0.1 / 1 / 10e-6, 0.2, 0.2],
   #     [15e6, np.pi / 2, 10 / 1 / 10e-6, 2, 2],
   # ],
)
print(f"Fitted parameters: {popt}")
print('T2e time ' + str(1/popt[0]*1e6) + ' us') 

In [ ]:
# plot measurement results (XXX) 5 ms reset
fig = plt.figure()
plt.plot(echo_delay, echo_res, "k")
plt.plot(echo_delay, echo_res, ".k")
plt.ylabel("A (a.u.)")
plt.xlabel(r"delay (s)")

# increase number of plot points for smooth plotting of fit results
delay_plot = np.linspace(echo_delay[0], echo_delay[-1], 5 * len(echo_delay))

popt, pcov = exponential_decay.fit(
   echo_delay,
   echo_res,
   1/3.4e-6,
   -20,
   -20,
   plot=True,
   # bounds=[
   #     [0.01e6, -np.pi / 2, 0.1 / 1 / 10e-6, 0.2, 0.2],
   #     [15e6, np.pi / 2, 10 / 1 / 10e-6, 2, 2],
   # ],
)
print(f"Fitted parameters: {popt}")
print('T2e time ' + str(1/popt[0]*1e6) + ' us') 

In [ ]:
popt, pcov = exponential_decay.fit(
   echo_delay,
   np.angle(echo_res),
   1/3.4e-6,
   -20,
   -20,
   plot=True,
   # bounds=[
   #     [0.01e6, -np.pi / 2, 0.1 / 1 / 10e-6, 0.2, 0.2],
   #     [15e6, np.pi / 2, 10 / 1 / 10e-6, 2, 2],
   # ],
)
print(f"Fitted parameters: {popt}")
print('T2e time ' + str(1/popt[0]*1e6) + ' us') 